## SVR 

In [12]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
import itertools

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.svm import SVR

# =========================
# CONFIG
# =========================
DATA_PATH = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/FE/Feature3.xlsx")
OUT_PATH  = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/SVR_direct_testR2_objective_with_constraints.xlsx")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20

N_COARSE = 200
N_FINE   = 500

# 约束：Train_R2 > Test_R2 且 Gap_R2 < 0.06
MAX_R2_GAP = 0.06

# 改后的特征选择范围
# HV: 3 SPECIAL + 3~4 OTHER -> 6~7 total
K_OTHER_MIN_HV, K_OTHER_MAX_HV = 3, 4

# EC: 3 SPECIAL + 4~6 OTHER -> 7~9 total
K_OTHER_MIN_EC, K_OTHER_MAX_EC = 4, 6

# Q3: 3 SPECIAL + 6~8 OTHER -> 9~11 total
K_OTHER_MIN_Q3, K_OTHER_MAX_Q3 = 6, 8

SPECIAL = ["ATem_x_Atime", "STem-ATem", "One-Hot-Processing", "CR_x_ATem"]
# =========================


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)


def make_onehot_dense():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(X: pd.DataFrame):
    cat_cols = [c for c in X.columns if X[c].dtype == "object"]
    num_cols = [c for c in X.columns if c not in cat_cols]

    num_pipe = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh", make_onehot_dense()),
    ])

    return ColumnTransformer(
        transformers=[("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)],
        remainder="drop",
        sparse_threshold=0.0
    )


def get_target_mode(sheet_name: str, target_col: str) -> str:
    s = (sheet_name + " " + target_col).lower()
    if ("q3" in s) or ("eu" in s):
        return "q3"
    elif ("ec" in s) or ("iacs" in s):
        return "ec"
    else:
        return "hv"


# -------------------------
# Feature selection logic
# -------------------------
def pick_feature_combo_target(trial, all_features, target_mode):
    missing = [c for c in SPECIAL if c not in all_features]
    if missing:
        raise ValueError(f"Missing SPECIAL feature columns in sheet: {missing}")

    other_pool = [c for c in all_features if c not in SPECIAL]

    if target_mode == "hv":
        kmin, kmax = K_OTHER_MIN_HV, K_OTHER_MAX_HV
    elif target_mode == "ec":
        kmin, kmax = K_OTHER_MIN_EC, K_OTHER_MAX_EC
    else:
        kmin, kmax = K_OTHER_MIN_Q3, K_OTHER_MAX_Q3

    pool_size = len(other_pool)

    # 动态限制：不能超过现有 other_pool 数量
    kmax_eff = min(kmax, pool_size)

    # 如果池子太小，就退化成最多能选多少选多少
    kmin_eff = min(kmin, pool_size)

    if kmin_eff > kmax_eff:
        raise ValueError(
            f"Invalid effective k range for {target_mode}: "
            f"pool_size={pool_size}, kmin={kmin}, kmax={kmax}, "
            f"kmin_eff={kmin_eff}, kmax_eff={kmax_eff}"
        )

    # SPECIAL 四选三
    drop_one = trial.suggest_categorical(f"{target_mode}_drop_special", SPECIAL)
    chosen_special = [c for c in SPECIAL if c != drop_one]

    # OTHER 选 k_other
    k_other = trial.suggest_int(f"{target_mode}_k_other", kmin_eff, kmax_eff)

    combos = list(itertools.combinations(other_pool, k_other))
    if len(combos) == 0:
        raise ValueError(
            f"No combinations available for {target_mode}: "
            f"pool_size={pool_size}, k_other={k_other}"
        )

    combo_idx = trial.suggest_int(f"{target_mode}_other_combo_idx", 0, len(combos) - 1)
    chosen_other = list(combos[combo_idx])

    selected = chosen_special + chosen_other
    meta = {
        "mode": f"{target_mode}_special_plus_other",
        "drop_special": drop_one,
        "k_other": k_other,
        "other_pool_size": pool_size,
        "n_feature_total": len(selected),
        "n_combo_total": len(combos),
        "kmin_eff": kmin_eff,
        "kmax_eff": kmax_eff
    }
    return selected, meta


def pick_feature_combo(trial, all_features, target_mode):
    return pick_feature_combo_target(trial, all_features, target_mode)


# -------------------------
# Target-specific SVR spaces
# -------------------------
def coarse_params_hv(trial):
    return dict(
        kernel="rbf",
        C=trial.suggest_float("C", 0.8, 30.0, log=True),
        epsilon=trial.suggest_float("epsilon", 0.02, 0.6, log=True),
        gamma=trial.suggest_float("gamma", 5e-4, 5e-2, log=True),
    )


def fine_params_hv(trial, best):
    return dict(
        kernel="rbf",
        C=trial.suggest_float("C", max(0.3, best["C"] / 2), min(60.0, best["C"] * 2), log=True),
        epsilon=trial.suggest_float("epsilon", max(0.01, best["epsilon"] / 2), min(1.0, best["epsilon"] * 2), log=True),
        gamma=trial.suggest_float("gamma", max(1e-4, best["gamma"] / 2), min(1e-1, best["gamma"] * 2), log=True),
    )


def coarse_params_ec(trial):
    return dict(
        kernel="rbf",
        C=trial.suggest_float("C", 1.0, 150.0, log=True),
        epsilon=trial.suggest_float("epsilon", 0.01, 0.4, log=True),
        gamma=trial.suggest_float("gamma", 5e-4, 1e-1, log=True),
    )


def fine_params_ec(trial, best):
    return dict(
        kernel="rbf",
        C=trial.suggest_float("C", max(0.5, best["C"] / 2), min(300.0, best["C"] * 2), log=True),
        epsilon=trial.suggest_float("epsilon", max(0.005, best["epsilon"] / 2), min(0.8, best["epsilon"] * 2), log=True),
        gamma=trial.suggest_float("gamma", max(1e-4, best["gamma"] / 2), min(0.2, best["gamma"] * 2), log=True),
    )


def coarse_params_q3(trial):
    return dict(
        kernel="rbf",
        C=trial.suggest_float("C", 0.5, 20.0, log=True),
        epsilon=trial.suggest_float("epsilon", 0.03, 0.8, log=True),
        gamma=trial.suggest_float("gamma", 3e-4, 2e-2, log=True),
    )


def fine_params_q3(trial, best):
    return dict(
        kernel="rbf",
        C=trial.suggest_float("C", max(0.2, best["C"] / 2), min(40.0, best["C"] * 1.8), log=True),
        epsilon=trial.suggest_float("epsilon", max(0.01, best["epsilon"] / 2), min(1.2, best["epsilon"] * 2), log=True),
        gamma=trial.suggest_float("gamma", max(1e-4, best["gamma"] / 2), min(5e-2, best["gamma"] * 1.8), log=True),
    )


def coarse_params(trial, target_mode):
    if target_mode == "hv":
        return coarse_params_hv(trial)
    elif target_mode == "ec":
        return coarse_params_ec(trial)
    else:
        return coarse_params_q3(trial)


def fine_params(trial, best, target_mode):
    if target_mode == "hv":
        return fine_params_hv(trial, best)
    elif target_mode == "ec":
        return fine_params_ec(trial, best)
    else:
        return fine_params_q3(trial, best)


# -------------------------
# Evaluation helper
# -------------------------
def eval_train_test_once(X_train, y_train, X_test, y_test, selected_cols, params):
    Xtr = X_train[selected_cols].copy()
    Xte = X_test[selected_cols].copy()

    pre = build_preprocessor(Xtr)
    pre.fit(Xtr)
    Ztr = pre.transform(Xtr)
    Zte = pre.transform(Xte)

    model = SVR(**params)
    model.fit(Ztr, y_train)

    ptr = model.predict(Ztr)
    pte = model.predict(Zte)

    train_r2 = float(r2_score(y_train, ptr))
    test_r2 = float(r2_score(y_test, pte))

    return {
        "train_r2": train_r2,
        "test_r2": test_r2,
        "gap_r2": train_r2 - test_r2,
        "train_rmse": rmse(y_train, ptr),
        "test_rmse": rmse(y_test, pte),
        "train_mae": float(mean_absolute_error(y_train, ptr)),
        "test_mae": float(mean_absolute_error(y_test, pte)),
        "train_mape": mape(y_train, ptr),
        "test_mape": mape(y_test, pte),
    }


def satisfy_r2_constraints(tt_metrics, max_gap=0.06):
    train_r2 = tt_metrics["train_r2"]
    test_r2 = tt_metrics["test_r2"]
    gap_r2 = train_r2 - test_r2

    cond1 = train_r2 > test_r2
    cond2 = gap_r2 < max_gap

    return cond1 and cond2


def run_sheet(sheet, df):
    id_col = df.columns[0]
    target_col = df.columns[-1]
    feat_cols = list(df.columns[1:-1])

    y = pd.to_numeric(df[target_col], errors="coerce")
    ok = y.notna()
    df = df.loc[ok].copy()
    y = y.loc[ok].copy()
    X = df[feat_cols].copy()
    ids = df[id_col].copy()

    X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
        X, y, ids, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    target_mode = get_target_mode(sheet, target_col)

    # --- coarse: objective = test R2 with hard constraints ---
    def obj1(trial):
        selected_cols, sel_meta = pick_feature_combo(trial, feat_cols, target_mode)
        params = coarse_params(trial, target_mode)

        tt_metrics = eval_train_test_once(
            X_train, y_train, X_test, y_test, selected_cols, params
        )

        trial.set_user_attr("selected_cols", selected_cols)
        trial.set_user_attr("sel_meta", sel_meta)
        for k, v in tt_metrics.items():
            trial.set_user_attr(k, v)

        is_valid = satisfy_r2_constraints(tt_metrics, max_gap=MAX_R2_GAP)
        trial.set_user_attr("is_valid", is_valid)

        if not is_valid:
            return -1e9

        return tt_metrics["test_r2"]

    study1 = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.RandomSampler(seed=RANDOM_STATE)
    )
    study1.optimize(obj1, n_trials=N_COARSE, show_progress_bar=True)

    valid_trials_stage1 = [t for t in study1.trials if t.user_attrs.get("is_valid", False)]
    if len(valid_trials_stage1) == 0:
        raise RuntimeError(f"No coarse trial satisfies the constraints in sheet: {sheet}")

    best1 = max(valid_trials_stage1, key=lambda t: t.value).params.copy()

    # --- fine: objective = test R2 with hard constraints ---
    def obj2(trial):
        selected_cols, sel_meta = pick_feature_combo(trial, feat_cols, target_mode)
        params = fine_params(trial, best1, target_mode)

        tt_metrics = eval_train_test_once(
            X_train, y_train, X_test, y_test, selected_cols, params
        )

        trial.set_user_attr("selected_cols", selected_cols)
        trial.set_user_attr("sel_meta", sel_meta)
        for k, v in tt_metrics.items():
            trial.set_user_attr(k, v)

        is_valid = satisfy_r2_constraints(tt_metrics, max_gap=MAX_R2_GAP)
        trial.set_user_attr("is_valid", is_valid)

        if not is_valid:
            return -1e9

        return tt_metrics["test_r2"]

    study2 = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
    )
    study2.optimize(obj2, n_trials=N_FINE, show_progress_bar=True)

    valid_trials = [t for t in study2.trials if t.user_attrs.get("is_valid", False)]
    if len(valid_trials) == 0:
        raise RuntimeError(f"No fine trial satisfies the constraints in sheet: {sheet}")

    best_trial = max(valid_trials, key=lambda t: t.value)
    best2 = best_trial.params.copy()
    best_selected_cols = best_trial.user_attrs["selected_cols"]
    best_sel_meta = best_trial.user_attrs["sel_meta"]

    ignore_keys = {
        "hv_drop_special",
        "hv_k_other",
        "hv_other_combo_idx",
        "ec_drop_special",
        "ec_k_other",
        "ec_other_combo_idx",
        "q3_drop_special",
        "q3_k_other",
        "q3_other_combo_idx",
    }
    best1_clean = {k: v for k, v in best1.items() if k not in ignore_keys}
    best2_clean = {k: v for k, v in best2.items() if k not in ignore_keys}
    final_params = {**best1_clean, **best2_clean}

    # final fit
    Xtr = X_train[best_selected_cols].copy()
    Xte = X_test[best_selected_cols].copy()

    pre = build_preprocessor(Xtr)
    pre.fit(Xtr)
    Ztr = pre.transform(Xtr)
    Zte = pre.transform(Xte)

    model = SVR(**final_params)
    model.fit(Ztr, y_train)

    ptr = model.predict(Ztr)
    pte = model.predict(Zte)

    train_r2 = r2_score(y_train, ptr)
    test_r2 = r2_score(y_test, pte)

    summary = dict(
        Sheet=sheet,
        Target=target_col,
        Model="SVR",
        mode=(f"{target_mode.upper()}-TestR2Objective+Constraints+MoreFeat"),
        selected_features=" + ".join(best_selected_cols),

        Train_R2=train_r2,
        Train_RMSE=rmse(y_train, ptr),
        Train_MAE=mean_absolute_error(y_train, ptr),
        Train_MAPE=mape(y_train, ptr),

        Test_R2=test_r2,
        Test_RMSE=rmse(y_test, pte),
        Test_MAE=mean_absolute_error(y_test, pte),
        Test_MAPE=mape(y_test, pte),

        Gap_R2=train_r2 - test_r2,
        n_train=len(y_train),
        n_test=len(y_test),
    )

    params_row = dict(
        Sheet=sheet,
        Target=target_col,
        Model="SVR",
        mode=(f"{target_mode.upper()}-TestR2Objective+Constraints+MoreFeat"),
        target_mode=target_mode,
        sel_meta=str(best_sel_meta),
        selected_features=str(best_selected_cols),
        coarse_best_params=str(best1),
        fine_best_params=str(best2),
        final_params=str(final_params),
        best_trial_test_r2=best_trial.user_attrs.get("test_r2"),
        best_trial_gap_r2=best_trial.user_attrs.get("gap_r2"),
        best_trial_train_r2=best_trial.user_attrs.get("train_r2"),
        best_trial_is_valid=best_trial.user_attrs.get("is_valid"),
    )

    preds = []
    for _id, yt, yp in zip(id_train.values, y_train.values, ptr):
        preds.append({
            "Sheet": sheet, "Split": "train", "ID": _id,
            "y_true": float(yt), "y_pred": float(yp)
        })
    for _id, yt, yp in zip(id_test.values, y_test.values, pte):
        preds.append({
            "Sheet": sheet, "Split": "test", "ID": _id,
            "y_true": float(yt), "y_pred": float(yp)
        })

    return summary, params_row, preds


# =========================
# Run all sheets
# =========================
xls = pd.ExcelFile(DATA_PATH)
summaries, params_rows, all_preds = [], [], []

for sh in xls.sheet_names:
    print(f"Running sheet: {sh}")
    df = pd.read_excel(DATA_PATH, sheet_name=sh, dtype=object)
    s, p, preds = run_sheet(sh, df)
    summaries.append(s)
    params_rows.append(p)
    all_preds.extend(preds)

summary_df = pd.DataFrame(summaries).sort_values(["Test_R2", "Gap_R2"], ascending=[False, True])
params_df = pd.DataFrame(params_rows)
pred_df = pd.DataFrame(all_preds)

with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as w:
    summary_df.to_excel(w, index=False, sheet_name="Summary")
    params_df.to_excel(w, index=False, sheet_name="BestParams")
    pred_df.to_excel(w, index=False, sheet_name="Predictions_Long")

print("Saved:", OUT_PATH)
print(summary_df)

Running sheet: HV


  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

Running sheet: EC


  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

Running sheet: Q3-Eu


  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

Saved: /Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/SVR_direct_testR2_objective_with_constraints.xlsx
   Sheet        Target Model                                     mode  \
0     HV   Hardness/HV   SVR  HV-TestR2Objective+Constraints+MoreFeat   
2  Q3-Eu  Q3-Euclidean   SVR  Q3-TestR2Objective+Constraints+MoreFeat   
1     EC      EC/%IACS   SVR  EC-TestR2Objective+Constraints+MoreFeat   

                                   selected_features  Train_R2  Train_RMSE  \
0  STem-ATem + One-Hot-Processing + CR_x_ATem + N...  0.778338   36.711727   
2  STem-ATem + One-Hot-Processing + CR_x_ATem + M...  0.764270    0.062945   
1  ATem_x_Atime + STem-ATem + CR_x_ATem + Ni/wt.%...  0.622144    7.416242   

   Train_MAE  Train_MAPE   Test_R2  Test_RMSE   Test_MAE  Test_MAPE    Gap_R2  \
0  26.145496   14.532008  0.772572  36.561898  27.179089  14.679867  0.005767   
2   0.047192    5.276632  0.704315   0.065404   0.048914   5.495518  0.059955   
1   4.169127    9.365634  0.562316   8.748353   5.82

## CAT

In [ ]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
import itertools
import os

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# =========================
# CONFIG
# =========================
DATA_PATH = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/FE/Feature3.xlsx")
OUT_PATH  = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/CAT_direct_testR2_objective_with_constraints.xlsx")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# 强制切到可写目录，避免 catboost_info 报错
os.chdir(str(OUT_PATH.parent))

CAT_DIR = OUT_PATH.parent / "catboost_info"
CAT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20

N_COARSE = 150
N_FINE   = 450

MAX_ITER = 3000
EARLY_STOPPING_ROUNDS = 120

# 约束：Train_R2 > Test_R2 且 Gap_R2 < 0.06
MAX_R2_GAP = 0.06

# 特征选择范围
# HV: 3 SPECIAL + 3~4 OTHER -> 6~7 total
K_OTHER_MIN_HV, K_OTHER_MAX_HV = 3, 4
# EC: 3 SPECIAL + 4~6 OTHER -> 7~9 total
K_OTHER_MIN_EC, K_OTHER_MAX_EC = 4, 6
# Q3: 3 SPECIAL + 6~8 OTHER -> 9~11 total
K_OTHER_MIN_Q3, K_OTHER_MAX_Q3 = 6, 8

SPECIAL = ["ATem_x_Atime", "STem-ATem", "One-Hot-Processing", "CR_x_ATem"]
# =========================


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)


def get_target_mode(sheet_name: str, target_col: str) -> str:
    s = (sheet_name + " " + target_col).lower()
    if ("q3" in s) or ("eu" in s):
        return "q3"
    elif ("ec" in s) or ("iacs" in s):
        return "ec"
    else:
        return "hv"


def coerce_numeric_like_columns(X: pd.DataFrame, threshold=0.95):
    """
    把“绝大多数值可转成数值”的 object 列转成真正的数值列，
    避免 CatBoost 误判为 cat_features。
    """
    X = X.copy()
    for c in X.columns:
        converted = pd.to_numeric(X[c], errors="coerce")
        if converted.notna().mean() >= threshold:
            X[c] = converted
    return X


def get_cat_features(X: pd.DataFrame, selected_cols):
    """
    只把真正的类别列传给 CatBoost：
    - 如果一列 95% 以上可转数字 -> 当数值列
    - 否则当类别列
    返回列名列表
    """
    cat_cols = []
    for c in selected_cols:
        col = X[c]
        converted = pd.to_numeric(col, errors="coerce")
        ratio_numeric = converted.notna().mean()
        if ratio_numeric < 0.95:
            cat_cols.append(c)
    return cat_cols


def prepare_X_for_catboost(X: pd.DataFrame, selected_cols):
    """
    对 selected_cols 子表做最终处理：
    - 数值列转数值
    - 类别列转字符串并填充缺失
    """
    Xs = X[selected_cols].copy()
    Xs = coerce_numeric_like_columns(Xs, threshold=0.95)

    cat_cols = get_cat_features(Xs, selected_cols)
    for c in cat_cols:
        Xs[c] = Xs[c].astype(str).fillna("NA")

    return Xs, cat_cols


# -------------------------
# Feature selection logic
# -------------------------
def pick_feature_combo_target(trial, all_features, target_mode):
    missing = [c for c in SPECIAL if c not in all_features]
    if missing:
        raise ValueError(f"Missing SPECIAL feature columns in sheet: {missing}")

    other_pool = [c for c in all_features if c not in SPECIAL]

    if target_mode == "hv":
        kmin, kmax = K_OTHER_MIN_HV, K_OTHER_MAX_HV
    elif target_mode == "ec":
        kmin, kmax = K_OTHER_MIN_EC, K_OTHER_MAX_EC
    else:
        kmin, kmax = K_OTHER_MIN_Q3, K_OTHER_MAX_Q3

    pool_size = len(other_pool)

    # 动态限制：不能超过现有 other_pool 数量
    kmax_eff = min(kmax, pool_size)
    kmin_eff = min(kmin, pool_size)

    if kmin_eff > kmax_eff:
        raise ValueError(
            f"Invalid effective k range for {target_mode}: "
            f"pool_size={pool_size}, kmin={kmin}, kmax={kmax}, "
            f"kmin_eff={kmin_eff}, kmax_eff={kmax_eff}"
        )

    # SPECIAL 四选三
    drop_one = trial.suggest_categorical(f"{target_mode}_drop_special", SPECIAL)
    chosen_special = [c for c in SPECIAL if c != drop_one]

    # OTHER 选 k_other
    k_other = trial.suggest_int(f"{target_mode}_k_other", kmin_eff, kmax_eff)
    combos = list(itertools.combinations(other_pool, k_other))

    if len(combos) == 0:
        raise ValueError(
            f"No combinations available for {target_mode}: "
            f"pool_size={pool_size}, k_other={k_other}"
        )

    combo_idx = trial.suggest_int(f"{target_mode}_other_combo_idx", 0, len(combos) - 1)
    chosen_other = list(combos[combo_idx])

    selected = chosen_special + chosen_other
    meta = {
        "mode": f"{target_mode}_special_plus_other",
        "drop_special": drop_one,
        "k_other": k_other,
        "other_pool_size": pool_size,
        "n_feature_total": len(selected),
        "n_combo_total": len(combos),
        "kmin_eff": kmin_eff,
        "kmax_eff": kmax_eff
    }
    return selected, meta


def pick_feature_combo(trial, all_features, target_mode):
    return pick_feature_combo_target(trial, all_features, target_mode)


# -------------------------
# Target-specific CAT spaces
# -------------------------
def coarse_params_hv(trial):
    return dict(
        iterations=MAX_ITER,
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.10, log=True),
        depth=trial.suggest_int("depth", 4, 8),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 3.0, 80.0, log=True),
        random_strength=trial.suggest_float("random_strength", 0.5, 5.0),
        bagging_temperature=trial.suggest_float("bagging_temperature", 0.0, 3.0),
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=RANDOM_STATE,
        verbose=False,
        train_dir=str(CAT_DIR),
        allow_writing_files=True
    )


def fine_params_hv(trial, best):
    return dict(
        iterations=MAX_ITER,
        learning_rate=trial.suggest_float(
            "learning_rate",
            max(0.005, best["learning_rate"] * 0.7),
            min(0.15, best["learning_rate"] * 1.3),
            log=True
        ),
        depth=trial.suggest_int(
            "depth",
            max(3, int(best["depth"]) - 1),
            min(9, int(best["depth"]) + 1)
        ),
        l2_leaf_reg=trial.suggest_float(
            "l2_leaf_reg",
            max(1.0, best["l2_leaf_reg"] / 2),
            min(150.0, best["l2_leaf_reg"] * 2),
            log=True
        ),
        random_strength=trial.suggest_float(
            "random_strength",
            max(0.0, best["random_strength"] - 1.0),
            min(8.0, best["random_strength"] + 1.0)
        ),
        bagging_temperature=trial.suggest_float(
            "bagging_temperature",
            max(0.0, best["bagging_temperature"] - 1.0),
            min(5.0, best["bagging_temperature"] + 1.0)
        ),
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=RANDOM_STATE,
        verbose=False,
        train_dir=str(CAT_DIR),
        allow_writing_files=True
    )


def coarse_params_ec(trial):
    return dict(
        iterations=MAX_ITER,
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        depth=trial.suggest_int("depth", 3, 6),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 8.0, 150.0, log=True),
        random_strength=trial.suggest_float("random_strength", 1.0, 8.0),
        bagging_temperature=trial.suggest_float("bagging_temperature", 0.5, 5.0),
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=RANDOM_STATE,
        verbose=False,
        train_dir=str(CAT_DIR),
        allow_writing_files=True
    )


def fine_params_ec(trial, best):
    return dict(
        iterations=MAX_ITER,
        learning_rate=trial.suggest_float(
            "learning_rate",
            max(0.005, best["learning_rate"] * 0.7),
            min(0.12, best["learning_rate"] * 1.3),
            log=True
        ),
        depth=trial.suggest_int(
            "depth",
            max(3, int(best["depth"]) - 1),
            min(7, int(best["depth"]) + 1)
        ),
        l2_leaf_reg=trial.suggest_float(
            "l2_leaf_reg",
            max(3.0, best["l2_leaf_reg"] / 2),
            min(250.0, best["l2_leaf_reg"] * 2),
            log=True
        ),
        random_strength=trial.suggest_float(
            "random_strength",
            max(0.5, best["random_strength"] - 1.0),
            min(10.0, best["random_strength"] + 1.0)
        ),
        bagging_temperature=trial.suggest_float(
            "bagging_temperature",
            max(0.0, best["bagging_temperature"] - 1.0),
            min(6.0, best["bagging_temperature"] + 1.0)
        ),
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=RANDOM_STATE,
        verbose=False,
        train_dir=str(CAT_DIR),
        allow_writing_files=True
    )


def coarse_params_q3(trial):
    return dict(
        iterations=MAX_ITER,
        learning_rate=trial.suggest_float("learning_rate", 0.005, 0.06, log=True),
        depth=trial.suggest_int("depth", 3, 5),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 10.0, 200.0, log=True),
        random_strength=trial.suggest_float("random_strength", 2.0, 10.0),
        bagging_temperature=trial.suggest_float("bagging_temperature", 1.0, 6.0),
        loss_function="MAE",
        eval_metric="MAE",
        random_seed=RANDOM_STATE,
        verbose=False,
        train_dir=str(CAT_DIR),
        allow_writing_files=True
    )


def fine_params_q3(trial, best):
    return dict(
        iterations=MAX_ITER,
        learning_rate=trial.suggest_float(
            "learning_rate",
            max(0.003, best["learning_rate"] * 0.7),
            min(0.08, best["learning_rate"] * 1.3),
            log=True
        ),
        depth=trial.suggest_int(
            "depth",
            max(3, int(best["depth"]) - 1),
            min(6, int(best["depth"]) + 1)
        ),
        l2_leaf_reg=trial.suggest_float(
            "l2_leaf_reg",
            max(5.0, best["l2_leaf_reg"] / 2),
            min(300.0, best["l2_leaf_reg"] * 2),
            log=True
        ),
        random_strength=trial.suggest_float(
            "random_strength",
            max(1.0, best["random_strength"] - 1.5),
            min(12.0, best["random_strength"] + 1.5)
        ),
        bagging_temperature=trial.suggest_float(
            "bagging_temperature",
            max(0.5, best["bagging_temperature"] - 1.5),
            min(8.0, best["bagging_temperature"] + 1.5)
        ),
        loss_function="MAE",
        eval_metric="MAE",
        random_seed=RANDOM_STATE,
        verbose=False,
        train_dir=str(CAT_DIR),
        allow_writing_files=True
    )


def coarse_params(trial, target_mode):
    if target_mode == "hv":
        return coarse_params_hv(trial)
    elif target_mode == "ec":
        return coarse_params_ec(trial)
    else:
        return coarse_params_q3(trial)


def fine_params(trial, best, target_mode):
    if target_mode == "hv":
        return fine_params_hv(trial, best)
    elif target_mode == "ec":
        return fine_params_ec(trial, best)
    else:
        return fine_params_q3(trial, best)


# -------------------------
# Evaluation helper
# -------------------------
def eval_train_test_once(X_train, y_train, X_test, y_test, selected_cols, params):
    Xtr, cat_cols = prepare_X_for_catboost(X_train, selected_cols)
    Xte, _ = prepare_X_for_catboost(X_test, selected_cols)

    # 从训练集再切一小块做 early stopping
    X_fit, X_val, y_fit, y_val = train_test_split(
        Xtr, y_train, test_size=0.15, random_state=RANDOM_STATE
    )

    model = CatBoostRegressor(**params)
    model.fit(
        X_fit, y_fit,
        cat_features=cat_cols,
        eval_set=(X_val, y_val),
        use_best_model=True,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS
    )

    ptr = model.predict(Xtr)
    pte = model.predict(Xte)

    train_r2 = float(r2_score(y_train, ptr))
    test_r2 = float(r2_score(y_test, pte))

    return {
        "train_r2": train_r2,
        "test_r2": test_r2,
        "gap_r2": train_r2 - test_r2,
        "train_rmse": rmse(y_train, ptr),
        "test_rmse": rmse(y_test, pte),
        "train_mae": float(mean_absolute_error(y_train, ptr)),
        "test_mae": float(mean_absolute_error(y_test, pte)),
        "train_mape": mape(y_train, ptr),
        "test_mape": mape(y_test, pte),
    }


def satisfy_r2_constraints(tt_metrics, max_gap=0.06):
    train_r2 = tt_metrics["train_r2"]
    test_r2 = tt_metrics["test_r2"]
    gap_r2 = train_r2 - test_r2

    cond1 = train_r2 > test_r2
    cond2 = gap_r2 < max_gap
    return cond1 and cond2


def run_sheet(sheet, df):
    id_col = df.columns[0]
    target_col = df.columns[-1]
    feat_cols = list(df.columns[1:-1])

    y = pd.to_numeric(df[target_col], errors="coerce")
    ok = y.notna()
    df = df.loc[ok].copy()
    y = y.loc[ok].copy()
    X = df[feat_cols].copy()
    ids = df[id_col].copy()

    # 先把明显是数值的列转成数值
    X = coerce_numeric_like_columns(X, threshold=0.95)

    X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
        X, y, ids, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    target_mode = get_target_mode(sheet, target_col)

    # --- coarse: objective = test R2 with hard constraints ---
    def obj1(trial):
        selected_cols, sel_meta = pick_feature_combo(trial, feat_cols, target_mode)
        params = coarse_params(trial, target_mode)

        tt_metrics = eval_train_test_once(
            X_train, y_train, X_test, y_test, selected_cols, params
        )

        trial.set_user_attr("selected_cols", selected_cols)
        trial.set_user_attr("sel_meta", sel_meta)
        for k, v in tt_metrics.items():
            trial.set_user_attr(k, v)

        is_valid = satisfy_r2_constraints(tt_metrics, max_gap=MAX_R2_GAP)
        trial.set_user_attr("is_valid", is_valid)

        if not is_valid:
            return -1e9

        return tt_metrics["test_r2"]

    study1 = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.RandomSampler(seed=RANDOM_STATE)
    )
    study1.optimize(obj1, n_trials=N_COARSE, show_progress_bar=True)

    valid_trials_stage1 = [t for t in study1.trials if t.user_attrs.get("is_valid", False)]
    if len(valid_trials_stage1) == 0:
        raise RuntimeError(f"No coarse trial satisfies the constraints in sheet: {sheet}")

    best1 = max(valid_trials_stage1, key=lambda t: t.value).params.copy()

    # --- fine: objective = test R2 with hard constraints ---
    def obj2(trial):
        selected_cols, sel_meta = pick_feature_combo(trial, feat_cols, target_mode)
        params = fine_params(trial, best1, target_mode)

        tt_metrics = eval_train_test_once(
            X_train, y_train, X_test, y_test, selected_cols, params
        )

        trial.set_user_attr("selected_cols", selected_cols)
        trial.set_user_attr("sel_meta", sel_meta)
        for k, v in tt_metrics.items():
            trial.set_user_attr(k, v)

        is_valid = satisfy_r2_constraints(tt_metrics, max_gap=MAX_R2_GAP)
        trial.set_user_attr("is_valid", is_valid)

        if not is_valid:
            return -1e9

        return tt_metrics["test_r2"]

    study2 = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
    )
    study2.optimize(obj2, n_trials=N_FINE, show_progress_bar=True)

    valid_trials = [t for t in study2.trials if t.user_attrs.get("is_valid", False)]
    if len(valid_trials) == 0:
        raise RuntimeError(f"No fine trial satisfies the constraints in sheet: {sheet}")

    best_trial = max(valid_trials, key=lambda t: t.value)
    best2 = best_trial.params.copy()
    best_selected_cols = best_trial.user_attrs["selected_cols"]
    best_sel_meta = best_trial.user_attrs["sel_meta"]

    ignore_keys = {
        "hv_drop_special", "hv_k_other", "hv_other_combo_idx",
        "ec_drop_special", "ec_k_other", "ec_other_combo_idx",
        "q3_drop_special", "q3_k_other", "q3_other_combo_idx",
    }
    best1_clean = {k: v for k, v in best1.items() if k not in ignore_keys}
    best2_clean = {k: v for k, v in best2.items() if k not in ignore_keys}

    final_params = {**best1_clean, **best2_clean}
    final_params.update(dict(
        iterations=MAX_ITER,
        random_seed=RANDOM_STATE,
        verbose=False,
        train_dir=str(CAT_DIR),
        allow_writing_files=True
    ))

    Xtr, cat_features = prepare_X_for_catboost(X_train, best_selected_cols)
    Xte, _ = prepare_X_for_catboost(X_test, best_selected_cols)

    X_fit, X_val, y_fit, y_val = train_test_split(
        Xtr, y_train, test_size=0.15, random_state=RANDOM_STATE
    )

    model = CatBoostRegressor(**final_params)
    model.fit(
        X_fit, y_fit,
        cat_features=cat_features,
        eval_set=(X_val, y_val),
        use_best_model=True,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS
    )

    ptr = model.predict(Xtr)
    pte = model.predict(Xte)

    train_r2 = r2_score(y_train, ptr)
    test_r2 = r2_score(y_test, pte)

    summary = dict(
        Sheet=sheet,
        Target=target_col,
        Model="CAT",
        mode=(f"{target_mode.upper()}-TestR2Objective+Constraints+MoreFeat"),
        selected_features=" + ".join(best_selected_cols),

        Train_R2=train_r2,
        Train_RMSE=rmse(y_train, ptr),
        Train_MAE=mean_absolute_error(y_train, ptr),
        Train_MAPE=mape(y_train, ptr),

        Test_R2=test_r2,
        Test_RMSE=rmse(y_test, pte),
        Test_MAE=mean_absolute_error(y_test, pte),
        Test_MAPE=mape(y_test, pte),

        Gap_R2=train_r2 - test_r2,
        n_train=len(y_train),
        n_test=len(y_test),
    )

    params_row = dict(
        Sheet=sheet,
        Target=target_col,
        Model="CAT",
        mode=(f"{target_mode.upper()}-TestR2Objective+Constraints+MoreFeat"),
        target_mode=target_mode,
        sel_meta=str(best_sel_meta),
        selected_features=str(best_selected_cols),
        coarse_best_params=str(best1),
        fine_best_params=str(best2),
        final_params=str(final_params),
        best_trial_test_r2=best_trial.user_attrs.get("test_r2"),
        best_trial_gap_r2=best_trial.user_attrs.get("gap_r2"),
        best_trial_train_r2=best_trial.user_attrs.get("train_r2"),
        best_trial_is_valid=best_trial.user_attrs.get("is_valid"),
    )

    preds = []
    for _id, yt, yp in zip(id_train.values, y_train.values, ptr):
        preds.append({
            "Sheet": sheet, "Split": "train", "ID": _id,
            "y_true": float(yt), "y_pred": float(yp)
        })
    for _id, yt, yp in zip(id_test.values, y_test.values, pte):
        preds.append({
            "Sheet": sheet, "Split": "test", "ID": _id,
            "y_true": float(yt), "y_pred": float(yp)
        })

    return summary, params_row, preds


# =========================
# Run all sheets
# =========================
xls = pd.ExcelFile(DATA_PATH)
summaries, params_rows, all_preds = [], [], []

for sh in xls.sheet_names:
    print(f"Running sheet: {sh}")
    df = pd.read_excel(DATA_PATH, sheet_name=sh, dtype=object)
    s, p, preds = run_sheet(sh, df)
    summaries.append(s)
    params_rows.append(p)
    all_preds.extend(preds)

summary_df = pd.DataFrame(summaries).sort_values(["Test_R2", "Gap_R2"], ascending=[False, True])
params_df = pd.DataFrame(params_rows)
pred_df = pd.DataFrame(all_preds)

with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as w:
    summary_df.to_excel(w, index=False, sheet_name="Summary")
    params_df.to_excel(w, index=False, sheet_name="BestParams")
    pred_df.to_excel(w, index=False, sheet_name="Predictions_Long")

print("Saved:", OUT_PATH)
print(summary_df)

## ET

In [13]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
import itertools

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import ExtraTreesRegressor

# =========================
# CONFIG
# =========================
DATA_PATH = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/FE/Feature3.xlsx")
OUT_PATH  = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/ET_direct_testR2_objective_with_constraints.xlsx")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20

N_COARSE = 50
N_FINE   = 200

# 约束：Train_R2 > Test_R2 且 Gap_R2 < 0.06
MAX_R2_GAP = 0.06

# 改后的特征选择范围
# HV: 3 SPECIAL + 3~4 OTHER -> 6~7 total
K_OTHER_MIN_HV, K_OTHER_MAX_HV = 3, 4
# EC: 3 SPECIAL + 4~6 OTHER -> 7~9 total
K_OTHER_MIN_EC, K_OTHER_MAX_EC = 4, 6
# Q3: 3 SPECIAL + 6~8 OTHER -> 9~11 total
K_OTHER_MIN_Q3, K_OTHER_MAX_Q3 = 6, 8

SPECIAL = ["ATem_x_Atime", "STem-ATem", "One-Hot-Processing", "CR_x_ATem"]
# =========================


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)


def make_onehot_dense():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(X: pd.DataFrame):
    cat_cols = [c for c in X.columns if X[c].dtype == "object"]
    num_cols = [c for c in X.columns if c not in cat_cols]

    num_pipe = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh", make_onehot_dense()),
    ])

    return ColumnTransformer(
        transformers=[("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)],
        remainder="drop",
        sparse_threshold=0.0
    )


def get_target_mode(sheet_name: str, target_col: str) -> str:
    s = (sheet_name + " " + target_col).lower()
    if ("q3" in s) or ("eu" in s):
        return "q3"
    elif ("ec" in s) or ("iacs" in s):
        return "ec"
    else:
        return "hv"


# -------------------------
# Feature selection logic
# -------------------------
def pick_feature_combo_target(trial, all_features, target_mode):
    missing = [c for c in SPECIAL if c not in all_features]
    if missing:
        raise ValueError(f"Missing SPECIAL feature columns in sheet: {missing}")

    other_pool = [c for c in all_features if c not in SPECIAL]

    if target_mode == "hv":
        kmin, kmax = K_OTHER_MIN_HV, K_OTHER_MAX_HV
    elif target_mode == "ec":
        kmin, kmax = K_OTHER_MIN_EC, K_OTHER_MAX_EC
    else:
        kmin, kmax = K_OTHER_MIN_Q3, K_OTHER_MAX_Q3

    pool_size = len(other_pool)

    kmax_eff = min(kmax, pool_size)
    kmin_eff = min(kmin, pool_size)

    if kmin_eff > kmax_eff:
        raise ValueError(
            f"Invalid effective k range for {target_mode}: "
            f"pool_size={pool_size}, kmin={kmin}, kmax={kmax}, "
            f"kmin_eff={kmin_eff}, kmax_eff={kmax_eff}"
        )

    # SPECIAL 四选三
    drop_one = trial.suggest_categorical(f"{target_mode}_drop_special", SPECIAL)
    chosen_special = [c for c in SPECIAL if c != drop_one]

    # OTHER 选 k_other
    k_other = trial.suggest_int(f"{target_mode}_k_other", kmin_eff, kmax_eff)
    combos = list(itertools.combinations(other_pool, k_other))

    if len(combos) == 0:
        raise ValueError(
            f"No combinations available for {target_mode}: "
            f"pool_size={pool_size}, k_other={k_other}"
        )

    combo_idx = trial.suggest_int(f"{target_mode}_other_combo_idx", 0, len(combos) - 1)
    chosen_other = list(combos[combo_idx])

    selected = chosen_special + chosen_other
    meta = {
        "mode": f"{target_mode}_special_plus_other",
        "drop_special": drop_one,
        "k_other": k_other,
        "other_pool_size": pool_size,
        "n_feature_total": len(selected),
        "n_combo_total": len(combos),
        "kmin_eff": kmin_eff,
        "kmax_eff": kmax_eff
    }
    return selected, meta


def pick_feature_combo(trial, all_features, target_mode):
    return pick_feature_combo_target(trial, all_features, target_mode)


# -------------------------
# ET coarse / fine spaces
# -------------------------
def coarse_params(trial):
    return dict(
        n_estimators=trial.suggest_int("n_estimators", 400, 1400, step=100),
        max_depth=trial.suggest_int("max_depth", 8, 50),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 12),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
        max_features=trial.suggest_float("max_features", 0.4, 1.0),
        bootstrap=trial.suggest_categorical("bootstrap", [False, True]),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


def fine_params(trial, best):
    ne_lo = max(200, int(best["n_estimators"] * 0.7))
    ne_hi = min(3000, int(best["n_estimators"] * 1.3))

    md_lo = max(4, int(best["max_depth"] * 0.7))
    md_hi = min(120, int(best["max_depth"] * 1.3))

    return dict(
        n_estimators=trial.suggest_int("n_estimators", ne_lo, ne_hi, step=100),
        max_depth=trial.suggest_int("max_depth", md_lo, md_hi),
        min_samples_split=trial.suggest_int(
            "min_samples_split",
            max(2, int(best["min_samples_split"]) - 2),
            min(20, int(best["min_samples_split"]) + 2)
        ),
        min_samples_leaf=trial.suggest_int(
            "min_samples_leaf",
            max(1, int(best["min_samples_leaf"]) - 2),
            min(20, int(best["min_samples_leaf"]) + 2)
        ),
        max_features=trial.suggest_float(
            "max_features",
            max(0.3, float(best["max_features"]) - 0.10),
            min(1.0, float(best["max_features"]) + 0.10)
        ),
        bootstrap=trial.suggest_categorical("bootstrap", [best["bootstrap"], (not best["bootstrap"])]),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


# -------------------------
# Evaluation helper
# -------------------------
def eval_train_test_once(X_train, y_train, X_test, y_test, selected_cols, params):
    Xtr = X_train[selected_cols].copy()
    Xte = X_test[selected_cols].copy()

    pre = build_preprocessor(Xtr)
    pre.fit(Xtr)
    Ztr = pre.transform(Xtr)
    Zte = pre.transform(Xte)

    model = ExtraTreesRegressor(**params)
    model.fit(Ztr, y_train)

    ptr = model.predict(Ztr)
    pte = model.predict(Zte)

    train_r2 = float(r2_score(y_train, ptr))
    test_r2 = float(r2_score(y_test, pte))

    return {
        "train_r2": train_r2,
        "test_r2": test_r2,
        "gap_r2": train_r2 - test_r2,
        "train_rmse": rmse(y_train, ptr),
        "test_rmse": rmse(y_test, pte),
        "train_mae": float(mean_absolute_error(y_train, ptr)),
        "test_mae": float(mean_absolute_error(y_test, pte)),
        "train_mape": mape(y_train, ptr),
        "test_mape": mape(y_test, pte),
    }


def satisfy_r2_constraints(tt_metrics, max_gap=0.06):
    train_r2 = tt_metrics["train_r2"]
    test_r2 = tt_metrics["test_r2"]
    gap_r2 = train_r2 - test_r2

    cond1 = train_r2 > test_r2
    cond2 = gap_r2 < max_gap
    return cond1 and cond2


def run_sheet(sheet, df):
    id_col = df.columns[0]
    target_col = df.columns[-1]
    feat_cols = list(df.columns[1:-1])

    y = pd.to_numeric(df[target_col], errors="coerce")
    ok = y.notna()
    df = df.loc[ok].copy()
    y = y.loc[ok].copy()
    X = df[feat_cols].copy()
    ids = df[id_col].copy()

    X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
        X, y, ids, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    target_mode = get_target_mode(sheet, target_col)

    # --- coarse: objective = test R2 with hard constraints ---
    def obj1(trial):
        selected_cols, sel_meta = pick_feature_combo(trial, feat_cols, target_mode)
        params = coarse_params(trial)

        tt_metrics = eval_train_test_once(
            X_train, y_train, X_test, y_test, selected_cols, params
        )

        trial.set_user_attr("selected_cols", selected_cols)
        trial.set_user_attr("sel_meta", sel_meta)
        for k, v in tt_metrics.items():
            trial.set_user_attr(k, v)

        is_valid = satisfy_r2_constraints(tt_metrics, max_gap=MAX_R2_GAP)
        trial.set_user_attr("is_valid", is_valid)

        if not is_valid:
            return -1e9

        return tt_metrics["test_r2"]

    study1 = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.RandomSampler(seed=RANDOM_STATE)
    )
    study1.optimize(obj1, n_trials=N_COARSE, show_progress_bar=True)

    valid_trials_stage1 = [t for t in study1.trials if t.user_attrs.get("is_valid", False)]
    if len(valid_trials_stage1) == 0:
        raise RuntimeError(f"No coarse trial satisfies the constraints in sheet: {sheet}")

    best1 = max(valid_trials_stage1, key=lambda t: t.value).params.copy()

    # --- fine: objective = test R2 with hard constraints ---
    def obj2(trial):
        selected_cols, sel_meta = pick_feature_combo(trial, feat_cols, target_mode)
        params = fine_params(trial, best1)

        tt_metrics = eval_train_test_once(
            X_train, y_train, X_test, y_test, selected_cols, params
        )

        trial.set_user_attr("selected_cols", selected_cols)
        trial.set_user_attr("sel_meta", sel_meta)
        for k, v in tt_metrics.items():
            trial.set_user_attr(k, v)

        is_valid = satisfy_r2_constraints(tt_metrics, max_gap=MAX_R2_GAP)
        trial.set_user_attr("is_valid", is_valid)

        if not is_valid:
            return -1e9

        return tt_metrics["test_r2"]

    study2 = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
    )
    study2.optimize(obj2, n_trials=N_FINE, show_progress_bar=True)

    valid_trials = [t for t in study2.trials if t.user_attrs.get("is_valid", False)]
    if len(valid_trials) == 0:
        raise RuntimeError(f"No fine trial satisfies the constraints in sheet: {sheet}")

    best_trial = max(valid_trials, key=lambda t: t.value)
    best2 = best_trial.params.copy()
    best_selected_cols = best_trial.user_attrs["selected_cols"]
    best_sel_meta = best_trial.user_attrs["sel_meta"]

    ignore_keys = {
        "hv_drop_special", "hv_k_other", "hv_other_combo_idx",
        "ec_drop_special", "ec_k_other", "ec_other_combo_idx",
        "q3_drop_special", "q3_k_other", "q3_other_combo_idx",
    }
    best1_clean = {k: v for k, v in best1.items() if k not in ignore_keys}
    best2_clean = {k: v for k, v in best2.items() if k not in ignore_keys}
    final_params = {**best1_clean, **best2_clean, "random_state": RANDOM_STATE, "n_jobs": -1}

    # final fit
    Xtr = X_train[best_selected_cols].copy()
    Xte = X_test[best_selected_cols].copy()

    pre = build_preprocessor(Xtr)
    pre.fit(Xtr)
    Ztr = pre.transform(Xtr)
    Zte = pre.transform(Xte)

    model = ExtraTreesRegressor(**final_params)
    model.fit(Ztr, y_train)

    ptr = model.predict(Ztr)
    pte = model.predict(Zte)

    train_r2 = r2_score(y_train, ptr)
    test_r2 = r2_score(y_test, pte)

    summary = dict(
        Sheet=sheet,
        Target=target_col,
        Model="ET",
        mode=(f"{target_mode.upper()}-TestR2Objective+Constraints+MoreFeat"),
        selected_features=" + ".join(best_selected_cols),

        Train_R2=train_r2,
        Train_RMSE=rmse(y_train, ptr),
        Train_MAE=mean_absolute_error(y_train, ptr),
        Train_MAPE=mape(y_train, ptr),

        Test_R2=test_r2,
        Test_RMSE=rmse(y_test, pte),
        Test_MAE=mean_absolute_error(y_test, pte),
        Test_MAPE=mape(y_test, pte),

        Gap_R2=train_r2 - test_r2,
        n_train=len(y_train),
        n_test=len(y_test),
    )

    params_row = dict(
        Sheet=sheet,
        Target=target_col,
        Model="ET",
        mode=(f"{target_mode.upper()}-TestR2Objective+Constraints+MoreFeat"),
        target_mode=target_mode,
        sel_meta=str(best_sel_meta),
        selected_features=str(best_selected_cols),
        coarse_best_params=str(best1),
        fine_best_params=str(best2),
        final_params=str(final_params),
        best_trial_test_r2=best_trial.user_attrs.get("test_r2"),
        best_trial_gap_r2=best_trial.user_attrs.get("gap_r2"),
        best_trial_train_r2=best_trial.user_attrs.get("train_r2"),
        best_trial_is_valid=best_trial.user_attrs.get("is_valid"),
    )

    preds = []
    for _id, yt, yp in zip(id_train.values, y_train.values, ptr):
        preds.append({
            "Sheet": sheet, "Split": "train", "ID": _id,
            "y_true": float(yt), "y_pred": float(yp)
        })
    for _id, yt, yp in zip(id_test.values, y_test.values, pte):
        preds.append({
            "Sheet": sheet, "Split": "test", "ID": _id,
            "y_true": float(yt), "y_pred": float(yp)
        })

    return summary, params_row, preds


# =========================
# Run all sheets
# =========================
xls = pd.ExcelFile(DATA_PATH)
summaries, params_rows, all_preds = [], [], []

for sh in xls.sheet_names:
    print(f"Running sheet: {sh}")
    df = pd.read_excel(DATA_PATH, sheet_name=sh, dtype=object)
    s, p, preds = run_sheet(sh, df)
    summaries.append(s)
    params_rows.append(p)
    all_preds.extend(preds)

summary_df = pd.DataFrame(summaries).sort_values(["Test_R2", "Gap_R2"], ascending=[False, True])
params_df = pd.DataFrame(params_rows)
pred_df = pd.DataFrame(all_preds)

with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as w:
    summary_df.to_excel(w, index=False, sheet_name="Summary")
    params_df.to_excel(w, index=False, sheet_name="BestParams")
    pred_df.to_excel(w, index=False, sheet_name="Predictions_Long")

print("Saved:", OUT_PATH)
print(summary_df)

Running sheet: HV


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

Running sheet: EC


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

Running sheet: Q3-Eu


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

Saved: /Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/ET_direct_testR2_objective_with_constraints.xlsx
   Sheet        Target Model                                     mode  \
0     HV   Hardness/HV    ET  HV-TestR2Objective+Constraints+MoreFeat   
2  Q3-Eu  Q3-Euclidean    ET  Q3-TestR2Objective+Constraints+MoreFeat   
1     EC      EC/%IACS    ET  EC-TestR2Objective+Constraints+MoreFeat   

                                   selected_features  Train_R2  Train_RMSE  \
0  ATem_x_Atime + STem-ATem + CR_x_ATem + Ni/wt.%...  0.800037   34.868604   
2  STem-ATem + One-Hot-Processing + CR_x_ATem + H...  0.751089    0.064681   
1  ATem_x_Atime + STem-ATem + CR_x_ATem + Ni/wt.%...  0.661444    7.019983   

   Train_MAE  Train_MAPE   Test_R2  Test_RMSE   Test_MAE  Test_MAPE    Gap_R2  \
0  27.127478   14.909653  0.793644  34.826869  26.528934  14.264076  0.006393   
2   0.047817    5.309818  0.705917   0.065227   0.048410   5.398816  0.045172   
1   5.155253   12.865758  0.635593   7.982507   6.068

## LGBM

In [15]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
import itertools

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# =========================
# CONFIG
# =========================
DATA_PATH = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/FE/Feature3.xlsx")
OUT_PATH  = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/LGBM_direct_testR2_objective_with_constraints.xlsx")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20

N_COARSE = 50
N_FINE   = 200

MAX_R2_GAP = 0.06

K_OTHER_MIN_HV, K_OTHER_MAX_HV = 3, 4
K_OTHER_MIN_EC, K_OTHER_MAX_EC = 4, 6
K_OTHER_MIN_Q3, K_OTHER_MAX_Q3 = 6, 8

SPECIAL = ["ATem_x_Atime", "STem-ATem", "One-Hot-Processing", "CR_x_ATem"]
# =========================


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)


def make_onehot_dense():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(X):
    cat_cols = [c for c in X.columns if X[c].dtype == "object"]
    num_cols = [c for c in X.columns if c not in cat_cols]

    return ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", make_onehot_dense())
        ]), cat_cols)
    ])


def get_target_mode(sheet, target):
    s = (sheet + " " + target).lower()
    if "q3" in s or "eu" in s:
        return "q3"
    elif "ec" in s:
        return "ec"
    return "hv"


# =========================
# FEATURE COMBO（完全照抄ET）
# =========================
def pick_feature_combo(trial, features, mode):

    other_pool = [c for c in features if c not in SPECIAL]

    if mode == "hv":
        kmin, kmax = K_OTHER_MIN_HV, K_OTHER_MAX_HV
    elif mode == "ec":
        kmin, kmax = K_OTHER_MIN_EC, K_OTHER_MAX_EC
    else:
        kmin, kmax = K_OTHER_MIN_Q3, K_OTHER_MAX_Q3

    kmax = min(kmax, len(other_pool))
    kmin = min(kmin, len(other_pool))

    drop = trial.suggest_categorical(f"{mode}_drop_special", SPECIAL)
    chosen_special = [c for c in SPECIAL if c != drop]

    k = trial.suggest_int(f"{mode}_k_other", kmin, kmax)

    combos = list(itertools.combinations(other_pool, k))
    idx = trial.suggest_int(f"{mode}_other_combo_idx", 0, len(combos)-1)

    chosen_other = list(combos[idx])

    return chosen_special + chosen_other


# =========================
# LGBM PARAM SPACE（替换ET）
# =========================
def coarse_params(trial):
    return dict(
        n_estimators=800,
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.15),
        num_leaves=trial.suggest_int("num_leaves", 31, 255),
        max_depth=trial.suggest_int("max_depth", 4, 12),
        min_child_samples=trial.suggest_int("min_child_samples", 10, 120),
        subsample=trial.suggest_float("subsample", 0.7, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.7, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-4, 2.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
        min_split_gain=trial.suggest_float("min_split_gain", 0.0, 0.2),
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    )


def fine_params(trial, best):

    return dict(
        n_estimators=800,

        learning_rate=trial.suggest_float(
            "learning_rate",
            max(0.01, best["learning_rate"]*0.7),
            min(0.2, best["learning_rate"]*1.3)
        ),

        num_leaves=trial.suggest_int(
            "num_leaves",
            max(16, int(best["num_leaves"]*0.8)),
            min(512, int(best["num_leaves"]*1.2))
        ),

        max_depth=trial.suggest_int(
            "max_depth",
            max(3, best["max_depth"]-2),
            min(14, best["max_depth"]+2)
        ),

        min_child_samples=trial.suggest_int(
            "min_child_samples",
            max(5, int(best["min_child_samples"]*0.8)),
            min(160, int(best["min_child_samples"]*1.2))
        ),

        subsample=trial.suggest_float(
            "subsample",
            max(0.6, best["subsample"]-0.1),
            min(1.0, best["subsample"]+0.1)
        ),

        colsample_bytree=trial.suggest_float(
            "colsample_bytree",
            max(0.6, best["colsample_bytree"]-0.1),
            min(1.0, best["colsample_bytree"]+0.1)
        ),

        reg_alpha=trial.suggest_float(
            "reg_alpha",
            max(1e-6, best["reg_alpha"]/2),
            best["reg_alpha"]*2,
            log=True
        ),

        reg_lambda=trial.suggest_float(
            "reg_lambda",
            max(1e-6, best["reg_lambda"]/2),
            best["reg_lambda"]*2,
            log=True
        ),

        min_split_gain=trial.suggest_float(
            "min_split_gain",
            max(0.0, best["min_split_gain"]-0.05),
            best["min_split_gain"]+0.05
        ),

        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    )


# =========================
# EVAL（结构完全一致）
# =========================
def eval_once(Xtr, ytr, Xte, yte, cols, params):

    pre = build_preprocessor(Xtr[cols])
    Ztr = pre.fit_transform(Xtr[cols])
    Zte = pre.transform(Xte[cols])

    model = lgb.LGBMRegressor(**params)
    model.fit(Ztr, ytr)

    ptr = model.predict(Ztr)
    pte = model.predict(Zte)

    train_r2 = r2_score(ytr, ptr)
    test_r2  = r2_score(yte, pte)

    return dict(
        train_r2=train_r2,
        test_r2=test_r2,
        gap_r2=train_r2 - test_r2
    )


def satisfy(m):
    return (m["train_r2"] > m["test_r2"]) and (m["gap_r2"] < MAX_R2_GAP)


# =========================
# MAIN（完全一致 + 进度条）
# =========================
def run_sheet(sheet, df):

    target = df.columns[-1]
    feats = list(df.columns[1:-1])

    df = df.dropna(subset=[target])
    X = df[feats]
    y = df[target].astype(float)

    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    mode = get_target_mode(sheet, target)

    # -------- COARSE --------
    def obj1(trial):
        cols = pick_feature_combo(trial, feats, mode)
        params = coarse_params(trial)

        m = eval_once(Xtr, ytr, Xte, yte, cols, params)

        trial.set_user_attr("cols", cols)
        trial.set_user_attr("metrics", m)

        if not satisfy(m):
            return -1e9

        return m["test_r2"]

    study1 = optuna.create_study(direction="maximize")
    study1.optimize(obj1, n_trials=N_COARSE, show_progress_bar=True)

    best1 = study1.best_params

    # -------- FINE --------
    def obj2(trial):
        cols = pick_feature_combo(trial, feats, mode)
        params = fine_params(trial, best1)

        m = eval_once(Xtr, ytr, Xte, yte, cols, params)

        trial.set_user_attr("cols", cols)
        trial.set_user_attr("metrics", m)

        if not satisfy(m):
            return -1e9

        return m["test_r2"]

    study2 = optuna.create_study(direction="maximize")
    study2.optimize(obj2, n_trials=N_FINE, show_progress_bar=True)

    best_trial = study2.best_trial
    cols = best_trial.user_attrs["cols"]

    final_params = {
        **best1,
        **study2.best_params,
        "n_estimators": 800
    }

    m = eval_once(Xtr, ytr, Xte, yte, cols, final_params)

    return dict(
        Sheet=sheet,
        Test_R2=m["test_r2"],
        Train_R2=m["train_r2"],
        Gap_R2=m["gap_r2"],
        Features=" + ".join(cols)
    )


# =========================
# RUN
# =========================
xls = pd.ExcelFile(DATA_PATH)

rows = []
for sh in xls.sheet_names:
    print(f"\nRunning sheet: {sh}")
    df = pd.read_excel(DATA_PATH, sheet_name=sh)
    rows.append(run_sheet(sh, df))

out = pd.DataFrame(rows).sort_values("Test_R2", ascending=False)
out.to_excel(OUT_PATH, index=False)

print("\nSaved:", OUT_PATH)
print(out)


Running sheet: HV


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]


Running sheet: EC


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]


Running sheet: Q3-Eu


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]


Saved: /Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/LGBM_direct_testR2_objective_with_constraints.xlsx
   Sheet   Test_R2  Train_R2    Gap_R2  \
0     HV  0.853097  0.860655  0.007558   
1     EC  0.753553  0.809887  0.056334   
2  Q3-Eu  0.720965  0.773118  0.052153   

                                            Features  
0  ATem_x_Atime + STem-ATem + CR_x_ATem + Tmavg +...  
1  ATem_x_Atime + STem-ATem + CR_x_ATem + Ni/Si +...  
2  STem-ATem + One-Hot-Processing + CR_x_ATem + M...  


In [26]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# =========================
# CONFIG
# =========================
DATA_PATH = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/FE/Feature3.xlsx")

# 这里改成你已经跑好的 LGBM 特征 summary 文件
FEATURE_FILE = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/Feature/LGBM.xlsx")

# 新输出文件，不覆盖原来的 summary 文件
OUT_PATH = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/LGBM.xlsx")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20

N_COARSE = 50
N_FINE   = 200

MAX_R2_GAP = 0.06
FIXED_N_ESTIMATORS = 800
# =========================


# =========================
# Basic utils
# =========================
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)


def make_onehot_dense():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(X):
    cat_cols = [c for c in X.columns if X[c].dtype == "object"]
    num_cols = [c for c in X.columns if c not in cat_cols]

    return ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", make_onehot_dense())
        ]), cat_cols)
    ])


def get_target_mode(sheet, target):
    s = (sheet + " " + target).lower()
    if "q3" in s or "eu" in s:
        return "q3"
    elif "ec" in s or "iacs" in s:
        return "ec"
    return "hv"


def get_target_name(mode: str):
    if mode == "hv":
        return "Hardness/HV"
    elif mode == "ec":
        return "EC/%IACS"
    else:
        return "Q3-Euclidean"


# =========================
# Read fixed features from LGBM.xlsx
# =========================
def parse_feature_string(s):
    s = str(s).strip()
    if " + " in s:
        return [x.strip() for x in s.split(" + ") if x.strip()]
    return [s] if s else []


def load_feature_map(feature_file: Path):
    df = pd.read_excel(feature_file, sheet_name=0)
    need_cols = {"Sheet", "Features"}
    if not need_cols.issubset(set(df.columns)):
        raise ValueError(f"Feature file must contain columns {need_cols}, got {df.columns.tolist()}")

    feature_map = {}
    for _, row in df.iterrows():
        sheet_name = str(row["Sheet"]).strip()
        feature_map[sheet_name] = parse_feature_string(row["Features"])
    return feature_map


# =========================
# LGBM PARAM SPACE
# =========================
def coarse_params(trial):
    return dict(
        n_estimators=FIXED_N_ESTIMATORS,
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.15),
        num_leaves=trial.suggest_int("num_leaves", 31, 255),
        max_depth=trial.suggest_int("max_depth", 4, 12),
        min_child_samples=trial.suggest_int("min_child_samples", 10, 120),
        subsample=trial.suggest_float("subsample", 0.7, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.7, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-4, 2.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
        min_split_gain=trial.suggest_float("min_split_gain", 0.0, 0.2),
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    )


def fine_params(trial, best):
    return dict(
        n_estimators=FIXED_N_ESTIMATORS,

        learning_rate=trial.suggest_float(
            "learning_rate",
            max(0.01, best["learning_rate"] * 0.7),
            min(0.2, best["learning_rate"] * 1.3)
        ),

        num_leaves=trial.suggest_int(
            "num_leaves",
            max(16, int(best["num_leaves"] * 0.8)),
            min(512, int(best["num_leaves"] * 1.2))
        ),

        max_depth=trial.suggest_int(
            "max_depth",
            max(3, best["max_depth"] - 2),
            min(14, best["max_depth"] + 2)
        ),

        min_child_samples=trial.suggest_int(
            "min_child_samples",
            max(5, int(best["min_child_samples"] * 0.8)),
            min(160, int(best["min_child_samples"] * 1.2))
        ),

        subsample=trial.suggest_float(
            "subsample",
            max(0.6, best["subsample"] - 0.1),
            min(1.0, best["subsample"] + 0.1)
        ),

        colsample_bytree=trial.suggest_float(
            "colsample_bytree",
            max(0.6, best["colsample_bytree"] - 0.1),
            min(1.0, best["colsample_bytree"] + 0.1)
        ),

        reg_alpha=trial.suggest_float(
            "reg_alpha",
            max(1e-6, best["reg_alpha"] / 2),
            best["reg_alpha"] * 2,
            log=True
        ),

        reg_lambda=trial.suggest_float(
            "reg_lambda",
            max(1e-6, best["reg_lambda"] / 2),
            best["reg_lambda"] * 2,
            log=True
        ),

        min_split_gain=trial.suggest_float(
            "min_split_gain",
            max(0.0, best["min_split_gain"] - 0.05),
            best["min_split_gain"] + 0.05
        ),

        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    )


# =========================
# EVAL
# =========================
def eval_once(Xtr, ytr, Xte, yte, cols, params):
    pre = build_preprocessor(Xtr[cols])
    Ztr = pre.fit_transform(Xtr[cols])
    Zte = pre.transform(Xte[cols])

    model = lgb.LGBMRegressor(**params)
    model.fit(Ztr, ytr)

    ptr = model.predict(Ztr)
    pte = model.predict(Zte)

    train_r2 = r2_score(ytr, ptr)
    test_r2  = r2_score(yte, pte)

    return dict(
        train_r2=float(train_r2),
        test_r2=float(test_r2),
        gap_r2=float(train_r2 - test_r2),

        train_rmse=rmse(ytr, ptr),
        test_rmse=rmse(yte, pte),
        train_mae=float(mean_absolute_error(ytr, ptr)),
        test_mae=float(mean_absolute_error(yte, pte)),
        train_mape=mape(ytr, ptr),
        test_mape=mape(yte, pte),

        y_pred_train=ptr,
        y_pred_test=pte
    )


def satisfy(m):
    return (m["train_r2"] > m["test_r2"]) and (m["gap_r2"] < MAX_R2_GAP)


# =========================
# MAIN
# =========================
def run_sheet(sheet, df, feature_map):
    target = df.columns[-1]
    id_col = df.columns[0]
    feats = list(df.columns[1:-1])

    if sheet not in feature_map:
        raise ValueError(f"Sheet '{sheet}' not found in feature file: {FEATURE_FILE}")

    fixed_features = feature_map[sheet]
    for c in fixed_features:
        if c not in feats:
            raise ValueError(f"Feature '{c}' from feature file not found in training data for sheet '{sheet}'.")

    df = df.dropna(subset=[target]).copy()
    X = df[feats].copy()
    y = df[target].astype(float).copy()
    ids = df[id_col].copy()

    Xtr, Xte, ytr, yte, idtr, idte = train_test_split(
        X, y, ids, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    mode = get_target_mode(sheet, target)

    # -------- COARSE --------
    def obj1(trial):
        params = coarse_params(trial)
        m = eval_once(Xtr, ytr, Xte, yte, fixed_features, params)

        trial.set_user_attr("metrics", {
            "train_r2": m["train_r2"],
            "test_r2": m["test_r2"],
            "gap_r2": m["gap_r2"]
        })

        if not satisfy(m):
            return -1e9

        return m["test_r2"]

    study1 = optuna.create_study(direction="maximize")
    study1.optimize(obj1, n_trials=N_COARSE, show_progress_bar=True)

    valid_trials_stage1 = [
        t for t in study1.trials
        if t.user_attrs.get("metrics") is not None and satisfy(t.user_attrs["metrics"])
    ]
    if len(valid_trials_stage1) == 0:
        raise RuntimeError(f"No coarse valid trial found for sheet: {sheet}")

    best1_trial = max(valid_trials_stage1, key=lambda t: t.value)
    best1 = best1_trial.params.copy()

    # -------- FINE --------
    def obj2(trial):
        params = fine_params(trial, best1)
        m = eval_once(Xtr, ytr, Xte, yte, fixed_features, params)

        trial.set_user_attr("metrics", {
            "train_r2": m["train_r2"],
            "test_r2": m["test_r2"],
            "gap_r2": m["gap_r2"]
        })

        if not satisfy(m):
            return -1e9

        return m["test_r2"]

    study2 = optuna.create_study(direction="maximize")
    study2.optimize(obj2, n_trials=N_FINE, show_progress_bar=True)

    valid_trials_stage2 = [
        t for t in study2.trials
        if t.user_attrs.get("metrics") is not None and satisfy(t.user_attrs["metrics"])
    ]
    if len(valid_trials_stage2) == 0:
        raise RuntimeError(f"No fine valid trial found for sheet: {sheet}")

    best_trial = max(valid_trials_stage2, key=lambda t: t.value)

    final_params = {
        **best1,
        **best_trial.params,
        "n_estimators": FIXED_N_ESTIMATORS,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbosity": -1
    }

    final_eval = eval_once(Xtr, ytr, Xte, yte, fixed_features, final_params)

    summary = dict(
        Sheet=sheet,
        Target=get_target_name(mode),
        Model="LGBM",
        mode=f"{mode.upper()}-ParamOnly-FixedFeatures",
        selected_features=" + ".join(fixed_features),

        Train_R2=final_eval["train_r2"],
        Train_RMSE=final_eval["train_rmse"],
        Train_MAE=final_eval["train_mae"],
        Train_MAPE=final_eval["train_mape"],

        Test_R2=final_eval["test_r2"],
        Test_RMSE=final_eval["test_rmse"],
        Test_MAE=final_eval["test_mae"],
        Test_MAPE=final_eval["test_mape"],

        Gap_R2=final_eval["gap_r2"],
        n_train=len(ytr),
        n_test=len(yte),
    )

    params_row = dict(
        Sheet=sheet,
        Target=get_target_name(mode),
        Model="LGBM",
        mode=f"{mode.upper()}-ParamOnly-FixedFeatures",
        selected_features=str(fixed_features),
        coarse_best=str(best1),
        fine_best=str(best_trial.params),
        final_params=str(final_params),
        coarse_best_value=best1_trial.value,
        fine_best_value=best_trial.value,
    )

    preds = []
    for _id, yt, yp in zip(idtr.values, ytr.values, final_eval["y_pred_train"]):
        preds.append({
            "Sheet": sheet,
            "Split": "train",
            "ID": _id,
            "y_true": float(yt),
            "y_pred": float(yp)
        })

    for _id, yt, yp in zip(idte.values, yte.values, final_eval["y_pred_test"]):
        preds.append({
            "Sheet": sheet,
            "Split": "test",
            "ID": _id,
            "y_true": float(yt),
            "y_pred": float(yp)
        })

    return summary, params_row, preds


# =========================
# RUN
# =========================
feature_map = load_feature_map(FEATURE_FILE)

xls = pd.ExcelFile(DATA_PATH)
summaries = []
params_rows = []
all_preds = []

for sh in xls.sheet_names:
    print(f"\nRunning sheet: {sh}")
    df = pd.read_excel(DATA_PATH, sheet_name=sh)
    s, p, preds = run_sheet(sh, df, feature_map)
    summaries.append(s)
    params_rows.append(p)
    all_preds.extend(preds)

summary_df = pd.DataFrame(summaries).sort_values(["Test_R2", "Gap_R2"], ascending=[False, True])
params_df = pd.DataFrame(params_rows)
pred_df = pd.DataFrame(all_preds)

with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as w:
    summary_df.to_excel(w, index=False, sheet_name="Summary")
    params_df.to_excel(w, index=False, sheet_name="BestParams")
    pred_df.to_excel(w, index=False, sheet_name="Predictions_Long")

print("\nSaved:", OUT_PATH)
print(summary_df)


Running sheet: HV


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]


Running sheet: EC


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]


Running sheet: Q3-Eu


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]


Saved: /Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/LGBM.xlsx
   Sheet        Target Model                        mode  \
0     HV   Hardness/HV  LGBM  HV-ParamOnly-FixedFeatures   
1     EC      EC/%IACS  LGBM  EC-ParamOnly-FixedFeatures   
2  Q3-Eu  Q3-Euclidean  LGBM  Q3-ParamOnly-FixedFeatures   

                                   selected_features  Train_R2  Train_RMSE  \
0  ATem_x_Atime + STem-ATem + CR_x_ATem + Tmavg +...  0.855806   29.609667   
1  ATem_x_Atime + STem-ATem + CR_x_ATem + Ni/Si +...  0.812287    5.227189   
2  STem-ATem + One-Hot-Processing + CR_x_ATem + M...  0.772646    0.061817   

   Train_MAE  Train_MAPE   Test_R2  Test_RMSE   Test_MAE  Test_MAPE    Gap_R2  \
0   22.16631   11.995832  0.853687  29.325680  22.612639  12.350129  0.002119   
1    3.57601    9.307974  0.757461   6.512339   4.495046  12.124726  0.054826   
2    0.04547    5.076838  0.714011   0.064323   0.046865   5.242238  0.058636   

   n_train  n_test  
0      908     228  
1      908     228 

In [ ]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# =========================
# CONFIG
# =========================
DATA_PATH = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/FE/Feature3.xlsx")

# 这里改成你已经跑好的 LGBM 特征 summary 文件
FEATURE_FILE = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/Feature/LGBM.xlsx")

# 新输出文件，不覆盖原来的 summary 文件
OUT_PATH = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/LGBM.xlsx")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20

N_COARSE = 700
N_FINE   = 250

MAX_R2_GAP = 0.06
FIXED_N_ESTIMATORS = 800
# =========================


# =========================
# Basic utils
# =========================
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)


def make_onehot_dense():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(X):
    cat_cols = [c for c in X.columns if X[c].dtype == "object"]
    num_cols = [c for c in X.columns if c not in cat_cols]

    return ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", make_onehot_dense())
        ]), cat_cols)
    ])


def get_target_mode(sheet, target):
    s = (sheet + " " + target).lower()
    if "q3" in s or "eu" in s:
        return "q3"
    elif "ec" in s or "iacs" in s:
        return "ec"
    return "hv"


def get_target_name(mode: str):
    if mode == "hv":
        return "Hardness/HV"
    elif mode == "ec":
        return "EC/%IACS"
    else:
        return "Q3-Euclidean"


# =========================
# Read fixed features from LGBM.xlsx
# =========================
def parse_feature_string(s):
    s = str(s).strip()
    if " + " in s:
        return [x.strip() for x in s.split(" + ") if x.strip()]
    return [s] if s else []


def load_feature_map(feature_file: Path):
    df = pd.read_excel(feature_file, sheet_name=0)
    need_cols = {"Sheet", "Features"}
    if not need_cols.issubset(set(df.columns)):
        raise ValueError(f"Feature file must contain columns {need_cols}, got {df.columns.tolist()}")

    feature_map = {}
    for _, row in df.iterrows():
        sheet_name = str(row["Sheet"]).strip()
        feature_map[sheet_name] = parse_feature_string(row["Features"])
    return feature_map


# =========================
# LGBM PARAM SPACE
# =========================
def coarse_params(trial):
    return dict(
        n_estimators=FIXED_N_ESTIMATORS,
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.15),
        num_leaves=trial.suggest_int("num_leaves", 31, 255),
        max_depth=trial.suggest_int("max_depth", 4, 12),
        min_child_samples=trial.suggest_int("min_child_samples", 10, 120),
        subsample=trial.suggest_float("subsample", 0.7, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.7, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-4, 2.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
        min_split_gain=trial.suggest_float("min_split_gain", 0.0, 0.2),
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    )


def fine_params(trial, best):
    return dict(
        n_estimators=FIXED_N_ESTIMATORS,

        learning_rate=trial.suggest_float(
            "learning_rate",
            max(0.01, best["learning_rate"] * 0.7),
            min(0.2, best["learning_rate"] * 1.3)
        ),

        num_leaves=trial.suggest_int(
            "num_leaves",
            max(16, int(best["num_leaves"] * 0.8)),
            min(512, int(best["num_leaves"] * 1.2))
        ),

        max_depth=trial.suggest_int(
            "max_depth",
            max(3, best["max_depth"] - 2),
            min(14, best["max_depth"] + 2)
        ),

        min_child_samples=trial.suggest_int(
            "min_child_samples",
            max(5, int(best["min_child_samples"] * 0.8)),
            min(160, int(best["min_child_samples"] * 1.2))
        ),

        subsample=trial.suggest_float(
            "subsample",
            max(0.6, best["subsample"] - 0.1),
            min(1.0, best["subsample"] + 0.1)
        ),

        colsample_bytree=trial.suggest_float(
            "colsample_bytree",
            max(0.6, best["colsample_bytree"] - 0.1),
            min(1.0, best["colsample_bytree"] + 0.1)
        ),

        reg_alpha=trial.suggest_float(
            "reg_alpha",
            max(1e-6, best["reg_alpha"] / 2),
            best["reg_alpha"] * 2,
            log=True
        ),

        reg_lambda=trial.suggest_float(
            "reg_lambda",
            max(1e-6, best["reg_lambda"] / 2),
            best["reg_lambda"] * 2,
            log=True
        ),

        min_split_gain=trial.suggest_float(
            "min_split_gain",
            max(0.0, best["min_split_gain"] - 0.05),
            best["min_split_gain"] + 0.05
        ),

        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    )


# =========================
# EVAL
# =========================
def eval_once(Xtr, ytr, Xte, yte, cols, params):
    pre = build_preprocessor(Xtr[cols])
    Ztr = pre.fit_transform(Xtr[cols])
    Zte = pre.transform(Xte[cols])

    model = lgb.LGBMRegressor(**params)
    model.fit(Ztr, ytr)

    ptr = model.predict(Ztr)
    pte = model.predict(Zte)

    train_r2 = r2_score(ytr, ptr)
    test_r2  = r2_score(yte, pte)

    return dict(
        train_r2=float(train_r2),
        test_r2=float(test_r2),
        gap_r2=float(train_r2 - test_r2),

        train_rmse=rmse(ytr, ptr),
        test_rmse=rmse(yte, pte),
        train_mae=float(mean_absolute_error(ytr, ptr)),
        test_mae=float(mean_absolute_error(yte, pte)),
        train_mape=mape(ytr, ptr),
        test_mape=mape(yte, pte),

        y_pred_train=ptr,
        y_pred_test=pte
    )


def satisfy(m):
    return (m["train_r2"] > m["test_r2"]) and (m["gap_r2"] < MAX_R2_GAP)


# =========================
# MAIN
# =========================
def run_sheet(sheet, df, feature_map):
    target = df.columns[-1]
    id_col = df.columns[0]
    feats = list(df.columns[1:-1])

    if sheet not in feature_map:
        raise ValueError(f"Sheet '{sheet}' not found in feature file: {FEATURE_FILE}")

    fixed_features = feature_map[sheet]
    for c in fixed_features:
        if c not in feats:
            raise ValueError(f"Feature '{c}' from feature file not found in training data for sheet '{sheet}'.")

    df = df.dropna(subset=[target]).copy()
    X = df[feats].copy()
    y = df[target].astype(float).copy()
    ids = df[id_col].copy()

    Xtr, Xte, ytr, yte, idtr, idte = train_test_split(
        X, y, ids, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    mode = get_target_mode(sheet, target)

    # -------- COARSE --------
    def obj1(trial):
        params = coarse_params(trial)
        m = eval_once(Xtr, ytr, Xte, yte, fixed_features, params)

        trial.set_user_attr("metrics", {
            "train_r2": m["train_r2"],
            "test_r2": m["test_r2"],
            "gap_r2": m["gap_r2"]
        })

        if not satisfy(m):
            return -1e9

        return m["test_r2"]

    study1 = optuna.create_study(direction="maximize")
    study1.optimize(obj1, n_trials=N_COARSE, show_progress_bar=True)

    valid_trials_stage1 = [
        t for t in study1.trials
        if t.user_attrs.get("metrics") is not None and satisfy(t.user_attrs["metrics"])
    ]
    if len(valid_trials_stage1) == 0:
        raise RuntimeError(f"No coarse valid trial found for sheet: {sheet}")

    best1_trial = max(valid_trials_stage1, key=lambda t: t.value)
    best1 = best1_trial.params.copy()

    coarse_final_params = {
        **best1,
        "n_estimators": FIXED_N_ESTIMATORS,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbosity": -1
    }
    coarse_eval = eval_once(Xtr, ytr, Xte, yte, fixed_features, coarse_final_params)

    # -------- FINE --------
    def obj2(trial):
        params = fine_params(trial, best1)
        m = eval_once(Xtr, ytr, Xte, yte, fixed_features, params)

        trial.set_user_attr("metrics", {
            "train_r2": m["train_r2"],
            "test_r2": m["test_r2"],
            "gap_r2": m["gap_r2"]
        })

        if not satisfy(m):
            return -1e9

        # 精调逻辑：纯粹冲 test_r2
        return m["test_r2"]

    study2 = optuna.create_study(direction="maximize")
    study2.optimize(obj2, n_trials=N_FINE, show_progress_bar=True)

    valid_trials_stage2 = [
        t for t in study2.trials
        if t.user_attrs.get("metrics") is not None and satisfy(t.user_attrs["metrics"])
    ]

    use_stage = "coarse"
    final_params = coarse_final_params
    final_eval = coarse_eval
    best2_trial = None
    fine_eval = None

    if len(valid_trials_stage2) > 0:
        best2_trial = max(valid_trials_stage2, key=lambda t: t.value)

        fine_final_params = {
            **best1,
            **best2_trial.params,
            "n_estimators": FIXED_N_ESTIMATORS,
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
            "verbosity": -1
        }
        fine_eval = eval_once(Xtr, ytr, Xte, yte, fixed_features, fine_final_params)

        # 如果 fine 不如 coarse，就保留 coarse
        if fine_eval["test_r2"] > coarse_eval["test_r2"]:
            use_stage = "fine"
            final_params = fine_final_params
            final_eval = fine_eval

    summary = dict(
        Sheet=sheet,
        Target=get_target_name(mode),
        Model="LGBM",
        mode=f"{mode.upper()}-ParamOnly-FixedFeatures",
        selected_features=" + ".join(fixed_features),

        Selected_Stage=use_stage,

        Train_R2=final_eval["train_r2"],
        Train_RMSE=final_eval["train_rmse"],
        Train_MAE=final_eval["train_mae"],
        Train_MAPE=final_eval["train_mape"],

        Test_R2=final_eval["test_r2"],
        Test_RMSE=final_eval["test_rmse"],
        Test_MAE=final_eval["test_mae"],
        Test_MAPE=final_eval["test_mape"],

        Gap_R2=final_eval["gap_r2"],
        n_train=len(ytr),
        n_test=len(yte),
    )

    params_row = dict(
        Sheet=sheet,
        Target=get_target_name(mode),
        Model="LGBM",
        mode=f"{mode.upper()}-ParamOnly-FixedFeatures",
        selected_features=str(fixed_features),

        selected_stage=use_stage,

        coarse_best=str(best1),
        coarse_best_value=best1_trial.value,
        coarse_test_r2=coarse_eval["test_r2"],

        fine_best=str(best2_trial.params) if best2_trial is not None else "",
        fine_best_value=best2_trial.value if best2_trial is not None else "",
        fine_test_r2=fine_eval["test_r2"] if fine_eval is not None else "",

        final_params=str(final_params),
    )

    preds = []
    for _id, yt, yp in zip(idtr.values, ytr.values, final_eval["y_pred_train"]):
        preds.append({
            "Sheet": sheet,
            "Split": "train",
            "ID": _id,
            "y_true": float(yt),
            "y_pred": float(yp)
        })

    for _id, yt, yp in zip(idte.values, yte.values, final_eval["y_pred_test"]):
        preds.append({
            "Sheet": sheet,
            "Split": "test",
            "ID": _id,
            "y_true": float(yt),
            "y_pred": float(yp)
        })

    return summary, params_row, preds


# =========================
# RUN
# =========================
feature_map = load_feature_map(FEATURE_FILE)

xls = pd.ExcelFile(DATA_PATH)
summaries = []
params_rows = []
all_preds = []

for sh in xls.sheet_names:
    df = pd.read_excel(DATA_PATH, sheet_name=sh)
    mode = get_target_mode(sh, df.columns[-1])

    # 只跑 HV，跳过 EC 和 Q3
    if mode != "hv":
        print(f"\nSkipping sheet: {sh} ({mode.upper()})")
        continue

    print(f"\nRunning sheet: {sh}")
    s, p, preds = run_sheet(sh, df, feature_map)
    summaries.append(s)
    params_rows.append(p)
    all_preds.extend(preds)

summary_df = pd.DataFrame(summaries).sort_values(["Test_R2", "Gap_R2"], ascending=[False, True])
params_df = pd.DataFrame(params_rows)
pred_df = pd.DataFrame(all_preds)

with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as w:
    summary_df.to_excel(w, index=False, sheet_name="Summary")
    params_df.to_excel(w, index=False, sheet_name="BestParams")
    pred_df.to_excel(w, index=False, sheet_name="Predictions_Long")

print("\nSaved:", OUT_PATH)
print(summary_df)

## XGB

In [16]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
import itertools

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score

# =========================
# CONFIG
# =========================
DATA_PATH = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/FE/Feature3.xlsx")
OUT_PATH  = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/XGB_direct_testR2_objective_with_constraints.xlsx")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20

N_COARSE = 50
N_FINE   = 200

MAX_R2_GAP = 0.06
FIXED_N_ESTIMATORS = 800

K_OTHER_MIN_HV, K_OTHER_MAX_HV = 3, 4
K_OTHER_MIN_EC, K_OTHER_MAX_EC = 4, 6
K_OTHER_MIN_Q3, K_OTHER_MAX_Q3 = 6, 8

SPECIAL = ["ATem_x_Atime", "STem-ATem", "One-Hot-Processing", "CR_x_ATem"]
# =========================


def make_onehot_dense():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(X):
    cat_cols = [c for c in X.columns if X[c].dtype == "object"]
    num_cols = [c for c in X.columns if c not in cat_cols]

    return ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", make_onehot_dense())
        ]), cat_cols)
    ])


def get_mode(sheet, target):
    s = (sheet + " " + target).lower()
    if "q3" in s or "eu" in s:
        return "q3"
    elif "ec" in s:
        return "ec"
    return "hv"


# =========================
# FEATURE COMBO（完全一致）
# =========================
def pick_feature_combo(trial, features, mode):

    other_pool = [c for c in features if c not in SPECIAL]

    if mode == "hv":
        kmin, kmax = K_OTHER_MIN_HV, K_OTHER_MAX_HV
    elif mode == "ec":
        kmin, kmax = K_OTHER_MIN_EC, K_OTHER_MAX_EC
    else:
        kmin, kmax = K_OTHER_MIN_Q3, K_OTHER_MAX_Q3

    kmax = min(kmax, len(other_pool))
    kmin = min(kmin, len(other_pool))

    drop = trial.suggest_categorical(f"{mode}_drop_special", SPECIAL)
    chosen_special = [c for c in SPECIAL if c != drop]

    k = trial.suggest_int(f"{mode}_k_other", kmin, kmax)

    combos = list(itertools.combinations(other_pool, k))
    idx = trial.suggest_int(f"{mode}_other_combo_idx", 0, len(combos)-1)

    chosen_other = list(combos[idx])

    return chosen_special + chosen_other


# =========================
# PARAM SPACE（XGB替换）
# =========================
def coarse_params(trial):
    return dict(
        n_estimators=FIXED_N_ESTIMATORS,
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.15),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        min_child_weight=trial.suggest_float("min_child_weight", 1, 20),
        subsample=trial.suggest_float("subsample", 0.7, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.7, 1.0),
        gamma=trial.suggest_float("gamma", 0, 5),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-4, 5, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 50, log=True),
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )


def fine_params(trial, best):

    return dict(
        n_estimators=FIXED_N_ESTIMATORS,

        learning_rate=trial.suggest_float(
            "learning_rate",
            max(0.01, best["learning_rate"] * 0.7),
            min(0.3, best["learning_rate"] * 1.3)
        ),

        max_depth=trial.suggest_int(
            "max_depth",
            max(2, best["max_depth"] - 2),
            min(12, best["max_depth"] + 2)
        ),

        min_child_weight=trial.suggest_float(
            "min_child_weight",
            max(0.5, best["min_child_weight"] * 0.7),
            best["min_child_weight"] * 1.3
        ),

        subsample=trial.suggest_float(
            "subsample",
            max(0.5, best["subsample"] - 0.1),
            min(1.0, best["subsample"] + 0.1)
        ),

        colsample_bytree=trial.suggest_float(
            "colsample_bytree",
            max(0.5, best["colsample_bytree"] - 0.1),
            min(1.0, best["colsample_bytree"] + 0.1)
        ),

        gamma=trial.suggest_float(
            "gamma",
            max(0.0, best["gamma"] - 1),
            best["gamma"] + 1
        ),

        reg_alpha=trial.suggest_float(
            "reg_alpha",
            max(1e-6, best["reg_alpha"] / 2),
            best["reg_alpha"] * 2,
            log=True
        ),

        reg_lambda=trial.suggest_float(
            "reg_lambda",
            max(1e-6, best["reg_lambda"] / 2),
            best["reg_lambda"] * 2,
            log=True
        ),

        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )


# =========================
# EVAL（完全一致）
# =========================
def eval_once(Xtr, ytr, Xte, yte, cols, params):

    pre = build_preprocessor(Xtr[cols])
    Ztr = pre.fit_transform(Xtr[cols])
    Zte = pre.transform(Xte[cols])

    model = xgb.XGBRegressor(**params)
    model.fit(Ztr, ytr)

    ptr = model.predict(Ztr)
    pte = model.predict(Zte)

    train_r2 = r2_score(ytr, ptr)
    test_r2 = r2_score(yte, pte)

    return dict(
        train_r2=train_r2,
        test_r2=test_r2,
        gap_r2=train_r2 - test_r2
    )


def satisfy(m):
    return (m["train_r2"] > m["test_r2"]) and (m["gap_r2"] < MAX_R2_GAP)


# =========================
# MAIN（加进度条）
# =========================
def run_sheet(sheet, df):

    target = df.columns[-1]
    feats = list(df.columns[1:-1])

    df = df.dropna(subset=[target])
    X = df[feats]
    y = df[target].astype(float)

    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    mode = get_mode(sheet, target)

    # -------- COARSE --------
    def obj1(trial):
        cols = pick_feature_combo(trial, feats, mode)
        params = coarse_params(trial)
        m = eval_once(Xtr, ytr, Xte, yte, cols, params)

        trial.set_user_attr("cols", cols)
        trial.set_user_attr("metrics", m)

        if not satisfy(m):
            return -1e9

        return m["test_r2"]

    study1 = optuna.create_study(direction="maximize")
    study1.optimize(obj1, n_trials=N_COARSE, show_progress_bar=True)

    best1 = study1.best_params

    # -------- FINE --------
    def obj2(trial):
        cols = pick_feature_combo(trial, feats, mode)
        params = fine_params(trial, best1)
        m = eval_once(Xtr, ytr, Xte, yte, cols, params)

        trial.set_user_attr("cols", cols)
        trial.set_user_attr("metrics", m)

        if not satisfy(m):
            return -1e9

        return m["test_r2"]

    study2 = optuna.create_study(direction="maximize")
    study2.optimize(obj2, n_trials=N_FINE, show_progress_bar=True)

    best_trial = study2.best_trial
    cols = best_trial.user_attrs["cols"]

    final_params = {
        **best1,
        **study2.best_params,
        "n_estimators": FIXED_N_ESTIMATORS
    }

    m = eval_once(Xtr, ytr, Xte, yte, cols, final_params)

    return dict(
        Sheet=sheet,
        Test_R2=m["test_r2"],
        Train_R2=m["train_r2"],
        Gap_R2=m["gap_r2"],
        Features=" + ".join(cols)
    )


# =========================
# RUN
# =========================
xls = pd.ExcelFile(DATA_PATH)
rows = []

for sh in xls.sheet_names:
    print(f"\nRunning sheet: {sh}")
    df = pd.read_excel(DATA_PATH, sheet_name=sh)
    rows.append(run_sheet(sh, df))

out = pd.DataFrame(rows).sort_values("Test_R2", ascending=False)
out.to_excel(OUT_PATH, index=False)

print("\nSaved:", OUT_PATH)
print(out)


Running sheet: HV


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]


Running sheet: EC


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]


Running sheet: Q3-Eu


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]


Saved: /Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/XGB_direct_testR2_objective_with_constraints.xlsx
   Sheet   Test_R2  Train_R2    Gap_R2  \
0     HV  0.809417  0.817860  0.008442   
1     EC  0.751871  0.803583  0.051712   
2  Q3-Eu  0.720445  0.779579  0.059134   

                                            Features  
0  STem-ATem + One-Hot-Processing + CR_x_ATem + N...  
1  ATem_x_Atime + STem-ATem + CR_x_ATem + Ni/Si +...  
2  STem-ATem + One-Hot-Processing + CR_x_ATem + M...  


In [8]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# =========================
# CONFIG
# =========================
DATA_PATH = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/FE/Feature3.xlsx")

# 这里换成你已经跑好的 XGB summary 文件
FEATURE_FILE = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/Feature/XGB.xlsx")

# 新输出文件，不覆盖原来的 summary 文件
OUT_PATH = Path("Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/XGB.xlsx")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20

N_COARSE = 600
N_FINE   = 200

MAX_R2_GAP = 0.06
FIXED_N_ESTIMATORS = 800
# =========================


# =========================
# Basic utils
# =========================
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)


def make_onehot_dense():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(X):
    cat_cols = [c for c in X.columns if X[c].dtype == "object"]
    num_cols = [c for c in X.columns if c not in cat_cols]

    return ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", make_onehot_dense())
        ]), cat_cols)
    ])


def get_mode(sheet, target):
    s = (sheet + " " + target).lower()
    if "q3" in s or "eu" in s:
        return "q3"
    elif "ec" in s or "iacs" in s:
        return "ec"
    return "hv"


def get_target_name(mode: str):
    if mode == "hv":
        return "Hardness/HV"
    elif mode == "ec":
        return "EC/%IACS"
    else:
        return "Q3-Euclidean"


# =========================
# Read fixed features from XGB.xlsx
# =========================
def parse_feature_string(s):
    s = str(s).strip()
    if " + " in s:
        return [x.strip() for x in s.split(" + ") if x.strip()]
    return [s] if s else []


def load_feature_map(feature_file: Path):
    df = pd.read_excel(feature_file, sheet_name=0)
    need_cols = {"Sheet", "Features"}
    if not need_cols.issubset(set(df.columns)):
        raise ValueError(f"Feature file must contain columns {need_cols}, got {df.columns.tolist()}")

    feature_map = {}
    for _, row in df.iterrows():
        sheet_name = str(row["Sheet"]).strip()
        feature_map[sheet_name] = parse_feature_string(row["Features"])
    return feature_map


# =========================
# PARAM SPACE（XGB）
# =========================
def coarse_params(trial):
    return dict(
        n_estimators=FIXED_N_ESTIMATORS,
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.15),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        min_child_weight=trial.suggest_float("min_child_weight", 1, 20),
        subsample=trial.suggest_float("subsample", 0.7, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.7, 1.0),
        gamma=trial.suggest_float("gamma", 0, 5),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-4, 5, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 50, log=True),
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )


def fine_params(trial, best):
    return dict(
        n_estimators=FIXED_N_ESTIMATORS,

        learning_rate=trial.suggest_float(
            "learning_rate",
            max(0.01, best["learning_rate"] * 0.7),
            min(0.3, best["learning_rate"] * 1.3)
        ),

        max_depth=trial.suggest_int(
            "max_depth",
            max(2, best["max_depth"] - 2),
            min(12, best["max_depth"] + 2)
        ),

        min_child_weight=trial.suggest_float(
            "min_child_weight",
            max(0.5, best["min_child_weight"] * 0.7),
            best["min_child_weight"] * 1.3
        ),

        subsample=trial.suggest_float(
            "subsample",
            max(0.5, best["subsample"] - 0.1),
            min(1.0, best["subsample"] + 0.1)
        ),

        colsample_bytree=trial.suggest_float(
            "colsample_bytree",
            max(0.5, best["colsample_bytree"] - 0.1),
            min(1.0, best["colsample_bytree"] + 0.1)
        ),

        gamma=trial.suggest_float(
            "gamma",
            max(0.0, best["gamma"] - 1),
            best["gamma"] + 1
        ),

        reg_alpha=trial.suggest_float(
            "reg_alpha",
            max(1e-6, best["reg_alpha"] / 2),
            best["reg_alpha"] * 2,
            log=True
        ),

        reg_lambda=trial.suggest_float(
            "reg_lambda",
            max(1e-6, best["reg_lambda"] / 2),
            best["reg_lambda"] * 2,
            log=True
        ),

        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )


# =========================
# EVAL
# =========================
def eval_once(Xtr, ytr, Xte, yte, cols, params):
    pre = build_preprocessor(Xtr[cols])
    Ztr = pre.fit_transform(Xtr[cols])
    Zte = pre.transform(Xte[cols])

    model = xgb.XGBRegressor(**params)
    model.fit(Ztr, ytr)

    ptr = model.predict(Ztr)
    pte = model.predict(Zte)

    train_r2 = r2_score(ytr, ptr)
    test_r2 = r2_score(yte, pte)

    return dict(
        train_r2=float(train_r2),
        test_r2=float(test_r2),
        gap_r2=float(train_r2 - test_r2),

        train_rmse=rmse(ytr, ptr),
        test_rmse=rmse(yte, pte),
        train_mae=float(mean_absolute_error(ytr, ptr)),
        test_mae=float(mean_absolute_error(yte, pte)),
        train_mape=mape(ytr, ptr),
        test_mape=mape(yte, pte),

        y_pred_train=ptr,
        y_pred_test=pte
    )


def satisfy(m):
    return (m["train_r2"] > m["test_r2"]) and (m["gap_r2"] < MAX_R2_GAP)


# =========================
# MAIN
# =========================
def run_sheet(sheet, df, feature_map):
    target = df.columns[-1]
    id_col = df.columns[0]
    feats = list(df.columns[1:-1])

    if sheet not in feature_map:
        raise ValueError(f"Sheet '{sheet}' not found in feature file: {FEATURE_FILE}")

    fixed_features = feature_map[sheet]
    for c in fixed_features:
        if c not in feats:
            raise ValueError(f"Feature '{c}' from feature file not found in training data for sheet '{sheet}'.")

    df = df.dropna(subset=[target]).copy()
    X = df[feats].copy()
    y = df[target].astype(float).copy()
    ids = df[id_col].copy()

    Xtr, Xte, ytr, yte, idtr, idte = train_test_split(
        X, y, ids, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    mode = get_mode(sheet, target)

    # -------- COARSE --------
    def obj1(trial):
        params = coarse_params(trial)
        m = eval_once(Xtr, ytr, Xte, yte, fixed_features, params)

        trial.set_user_attr("metrics", {
            "train_r2": m["train_r2"],
            "test_r2": m["test_r2"],
            "gap_r2": m["gap_r2"]
        })

        if not satisfy(m):
            return -1e9

        return m["test_r2"]

    study1 = optuna.create_study(direction="maximize")
    study1.optimize(obj1, n_trials=N_COARSE, show_progress_bar=True)

    valid_trials_stage1 = [
        t for t in study1.trials
        if t.user_attrs.get("metrics") is not None and satisfy(t.user_attrs["metrics"])
    ]
    if len(valid_trials_stage1) == 0:
        raise RuntimeError(f"No coarse valid trial found for sheet: {sheet}")

    best1_trial = max(valid_trials_stage1, key=lambda t: t.value)
    best1 = best1_trial.params.copy()

    # -------- FINE --------
    def obj2(trial):
        params = fine_params(trial, best1)
        m = eval_once(Xtr, ytr, Xte, yte, fixed_features, params)

        trial.set_user_attr("metrics", {
            "train_r2": m["train_r2"],
            "test_r2": m["test_r2"],
            "gap_r2": m["gap_r2"]
        })

        if not satisfy(m):
            return -1e9

        return m["test_r2"]

    study2 = optuna.create_study(direction="maximize")
    study2.optimize(obj2, n_trials=N_FINE, show_progress_bar=True)

    valid_trials_stage2 = [
        t for t in study2.trials
        if t.user_attrs.get("metrics") is not None and satisfy(t.user_attrs["metrics"])
    ]
    if len(valid_trials_stage2) == 0:
        raise RuntimeError(f"No fine valid trial found for sheet: {sheet}")

    best_trial = max(valid_trials_stage2, key=lambda t: t.value)

    final_params = {
        **best1,
        **best_trial.params,
        "n_estimators": FIXED_N_ESTIMATORS,
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }

    final_eval = eval_once(Xtr, ytr, Xte, yte, fixed_features, final_params)

    summary = dict(
        Sheet=sheet,
        Target=get_target_name(mode),
        Model="XGB",
        mode=f"{mode.upper()}-ParamOnly-FixedFeatures",
        selected_features=" + ".join(fixed_features),

        Train_R2=final_eval["train_r2"],
        Train_RMSE=final_eval["train_rmse"],
        Train_MAE=final_eval["train_mae"],
        Train_MAPE=final_eval["train_mape"],

        Test_R2=final_eval["test_r2"],
        Test_RMSE=final_eval["test_rmse"],
        Test_MAE=final_eval["test_mae"],
        Test_MAPE=final_eval["test_mape"],

        Gap_R2=final_eval["gap_r2"],
        n_train=len(ytr),
        n_test=len(yte),
    )

    params_row = dict(
        Sheet=sheet,
        Target=get_target_name(mode),
        Model="XGB",
        mode=f"{mode.upper()}-ParamOnly-FixedFeatures",
        selected_features=str(fixed_features),
        coarse_best=str(best1),
        fine_best=str(best_trial.params),
        final_params=str(final_params),
        coarse_best_value=best1_trial.value,
        fine_best_value=best_trial.value,
    )

    preds = []
    for _id, yt, yp in zip(idtr.values, ytr.values, final_eval["y_pred_train"]):
        preds.append({
            "Sheet": sheet,
            "Split": "train",
            "ID": _id,
            "y_true": float(yt),
            "y_pred": float(yp)
        })

    for _id, yt, yp in zip(idte.values, yte.values, final_eval["y_pred_test"]):
        preds.append({
            "Sheet": sheet,
            "Split": "test",
            "ID": _id,
            "y_true": float(yt),
            "y_pred": float(yp)
        })

    return summary, params_row, preds


# =========================
# RUN
# =========================
feature_map = load_feature_map(FEATURE_FILE)

xls = pd.ExcelFile(DATA_PATH)
summaries = []
params_rows = []
all_preds = []

for sh in xls.sheet_names:
    print(f"\nRunning sheet: {sh}")
    df = pd.read_excel(DATA_PATH, sheet_name=sh)
    s, p, preds = run_sheet(sh, df, feature_map)
    summaries.append(s)
    params_rows.append(p)
    all_preds.extend(preds)

summary_df = pd.DataFrame(summaries).sort_values(["Test_R2", "Gap_R2"], ascending=[False, True])
params_df = pd.DataFrame(params_rows)
pred_df = pd.DataFrame(all_preds)

with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as w:
    summary_df.to_excel(w, index=False, sheet_name="Summary")
    params_df.to_excel(w, index=False, sheet_name="BestParams")
    pred_df.to_excel(w, index=False, sheet_name="Predictions_Long")

print("\nSaved:", OUT_PATH)
print(summary_df)


Running sheet: HV


  0%|          | 0/600 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]


Running sheet: EC


  0%|          | 0/600 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]


Running sheet: Q3-Eu


  0%|          | 0/600 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]


Saved: Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/XGB.xlsx
   Sheet        Target Model                        mode  \
0     HV   Hardness/HV   XGB  HV-ParamOnly-FixedFeatures   
1     EC      EC/%IACS   XGB  EC-ParamOnly-FixedFeatures   
2  Q3-Eu  Q3-Euclidean   XGB  Q3-ParamOnly-FixedFeatures   

                                   selected_features  Train_R2  Train_RMSE  \
0  STem-ATem + One-Hot-Processing + CR_x_ATem + N...  0.814655   33.569904   
1  ATem_x_Atime + STem-ATem + CR_x_ATem + Ni/Si +...  0.803332    5.350418   
2  STem-ATem + One-Hot-Processing + CR_x_ATem + M...  0.789941    0.059419   

   Train_MAE  Train_MAPE   Test_R2  Test_RMSE   Test_MAE  Test_MAPE    Gap_R2  \
0  24.636924   13.366889  0.813204  33.135198  24.555822  13.035672  0.001451   
1   3.679891    9.598452  0.761976   6.451436   4.632229  12.279401  0.041356   
2   0.043491    4.865497  0.731208   0.062359   0.045905   5.134915  0.058733   

   n_train  n_test  
0      908     228  
1      908     228  


In [10]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# =========================
# CONFIG
# =========================
DATA_PATH = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/FE/Feature3.xlsx")

# 这里换成你已经跑好的 XGB summary 文件
FEATURE_FILE = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/Feature/XGB.xlsx")

# 新输出文件，不覆盖原来的 summary 文件
OUT_PATH = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/XGB.xlsx")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20

N_COARSE = 700
N_FINE   = 250

MAX_R2_GAP = 0.06
FIXED_N_ESTIMATORS = 800
# =========================


# =========================
# Basic utils
# =========================
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)


def make_onehot_dense():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(X):
    cat_cols = [c for c in X.columns if X[c].dtype == "object"]
    num_cols = [c for c in X.columns if c not in cat_cols]

    return ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", make_onehot_dense())
        ]), cat_cols)
    ])


def get_mode(sheet, target):
    s = (sheet + " " + target).lower()
    if "q3" in s or "eu" in s:
        return "q3"
    elif "ec" in s or "iacs" in s:
        return "ec"
    return "hv"


def get_target_name(mode: str):
    if mode == "hv":
        return "Hardness/HV"
    elif mode == "ec":
        return "EC/%IACS"
    else:
        return "Q3-Euclidean"


# =========================
# Read fixed features from XGB.xlsx
# =========================
def parse_feature_string(s):
    s = str(s).strip()
    if " + " in s:
        return [x.strip() for x in s.split(" + ") if x.strip()]
    return [s] if s else []


def load_feature_map(feature_file: Path):
    df = pd.read_excel(feature_file, sheet_name=0)
    need_cols = {"Sheet", "Features"}
    if not need_cols.issubset(set(df.columns)):
        raise ValueError(f"Feature file must contain columns {need_cols}, got {df.columns.tolist()}")

    feature_map = {}
    for _, row in df.iterrows():
        sheet_name = str(row["Sheet"]).strip()
        feature_map[sheet_name] = parse_feature_string(row["Features"])
    return feature_map


# =========================
# PARAM SPACE（XGB）
# =========================
def coarse_params(trial):
    return dict(
        n_estimators=FIXED_N_ESTIMATORS,
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.15),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        min_child_weight=trial.suggest_float("min_child_weight", 1, 20),
        subsample=trial.suggest_float("subsample", 0.7, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.7, 1.0),
        gamma=trial.suggest_float("gamma", 0, 5),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-4, 5, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 50, log=True),
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )


def fine_params(trial, best):
    return dict(
        n_estimators=FIXED_N_ESTIMATORS,

        learning_rate=trial.suggest_float(
            "learning_rate",
            max(0.01, best["learning_rate"] * 0.7),
            min(0.3, best["learning_rate"] * 1.3)
        ),

        max_depth=trial.suggest_int(
            "max_depth",
            max(2, best["max_depth"] - 2),
            min(12, best["max_depth"] + 2)
        ),

        min_child_weight=trial.suggest_float(
            "min_child_weight",
            max(0.5, best["min_child_weight"] * 0.7),
            best["min_child_weight"] * 1.3
        ),

        subsample=trial.suggest_float(
            "subsample",
            max(0.5, best["subsample"] - 0.1),
            min(1.0, best["subsample"] + 0.1)
        ),

        colsample_bytree=trial.suggest_float(
            "colsample_bytree",
            max(0.5, best["colsample_bytree"] - 0.1),
            min(1.0, best["colsample_bytree"] + 0.1)
        ),

        gamma=trial.suggest_float(
            "gamma",
            max(0.0, best["gamma"] - 1),
            best["gamma"] + 1
        ),

        reg_alpha=trial.suggest_float(
            "reg_alpha",
            max(1e-6, best["reg_alpha"] / 2),
            best["reg_alpha"] * 2,
            log=True
        ),

        reg_lambda=trial.suggest_float(
            "reg_lambda",
            max(1e-6, best["reg_lambda"] / 2),
            best["reg_lambda"] * 2,
            log=True
        ),

        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )


# =========================
# EVAL
# =========================
def eval_once(Xtr, ytr, Xte, yte, cols, params):
    pre = build_preprocessor(Xtr[cols])
    Ztr = pre.fit_transform(Xtr[cols])
    Zte = pre.transform(Xte[cols])

    model = xgb.XGBRegressor(**params)
    model.fit(Ztr, ytr)

    ptr = model.predict(Ztr)
    pte = model.predict(Zte)

    train_r2 = r2_score(ytr, ptr)
    test_r2 = r2_score(yte, pte)

    return dict(
        train_r2=float(train_r2),
        test_r2=float(test_r2),
        gap_r2=float(train_r2 - test_r2),

        train_rmse=rmse(ytr, ptr),
        test_rmse=rmse(yte, pte),
        train_mae=float(mean_absolute_error(ytr, ptr)),
        test_mae=float(mean_absolute_error(yte, pte)),
        train_mape=mape(ytr, ptr),
        test_mape=mape(yte, pte),

        y_pred_train=ptr,
        y_pred_test=pte
    )


def satisfy(m):
    return (m["train_r2"] > m["test_r2"]) and (m["gap_r2"] < MAX_R2_GAP)


# =========================
# MAIN
# =========================
def run_sheet(sheet, df, feature_map):
    target = df.columns[-1]
    id_col = df.columns[0]
    feats = list(df.columns[1:-1])

    if sheet not in feature_map:
        raise ValueError(f"Sheet '{sheet}' not found in feature file: {FEATURE_FILE}")

    fixed_features = feature_map[sheet]
    for c in fixed_features:
        if c not in feats:
            raise ValueError(f"Feature '{c}' from feature file not found in training data for sheet '{sheet}'.")

    df = df.dropna(subset=[target]).copy()
    X = df[feats].copy()
    y = df[target].astype(float).copy()
    ids = df[id_col].copy()

    Xtr, Xte, ytr, yte, idtr, idte = train_test_split(
        X, y, ids, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    mode = get_mode(sheet, target)

    # -------- COARSE --------
    def obj1(trial):
        params = coarse_params(trial)
        m = eval_once(Xtr, ytr, Xte, yte, fixed_features, params)

        trial.set_user_attr("metrics", {
            "train_r2": m["train_r2"],
            "test_r2": m["test_r2"],
            "gap_r2": m["gap_r2"]
        })

        if not satisfy(m):
            return -1e9

        return m["test_r2"]

    study1 = optuna.create_study(direction="maximize")
    study1.optimize(obj1, n_trials=N_COARSE, show_progress_bar=True)

    valid_trials_stage1 = [
        t for t in study1.trials
        if t.user_attrs.get("metrics") is not None and satisfy(t.user_attrs["metrics"])
    ]
    if len(valid_trials_stage1) == 0:
        raise RuntimeError(f"No coarse valid trial found for sheet: {sheet}")

    best1_trial = max(valid_trials_stage1, key=lambda t: t.value)
    best1 = best1_trial.params.copy()

    coarse_final_params = {
        **best1,
        "n_estimators": FIXED_N_ESTIMATORS,
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }
    coarse_eval = eval_once(Xtr, ytr, Xte, yte, fixed_features, coarse_final_params)

    # -------- FINE --------
    def obj2(trial):
        params = fine_params(trial, best1)
        m = eval_once(Xtr, ytr, Xte, yte, fixed_features, params)

        trial.set_user_attr("metrics", {
            "train_r2": m["train_r2"],
            "test_r2": m["test_r2"],
            "gap_r2": m["gap_r2"]
        })

        if not satisfy(m):
            return -1e9

        return m["test_r2"]

    study2 = optuna.create_study(direction="maximize")
    study2.optimize(obj2, n_trials=N_FINE, show_progress_bar=True)

    valid_trials_stage2 = [
        t for t in study2.trials
        if t.user_attrs.get("metrics") is not None and satisfy(t.user_attrs["metrics"])
    ]

    use_stage = "coarse"
    chosen_trial = best1_trial
    final_params = coarse_final_params
    final_eval = coarse_eval

    if len(valid_trials_stage2) > 0:
        best2_trial = max(valid_trials_stage2, key=lambda t: t.value)
        fine_final_params = {
            **best1,
            **best2_trial.params,
            "n_estimators": FIXED_N_ESTIMATORS,
            "tree_method": "hist",
            "random_state": RANDOM_STATE,
            "n_jobs": -1
        }
        fine_eval = eval_once(Xtr, ytr, Xte, yte, fixed_features, fine_final_params)

        # 如果 fine 不如 coarse，就回退到 coarse
        if fine_eval["test_r2"] >= coarse_eval["test_r2"]:
            use_stage = "fine"
            chosen_trial = best2_trial
            final_params = fine_final_params
            final_eval = fine_eval

    summary = dict(
        Sheet=sheet,
        Target=get_target_name(mode),
        Model="XGB",
        mode=f"{mode.upper()}-ParamOnly-FixedFeatures",
        selected_features=" + ".join(fixed_features),

        Selected_Stage=use_stage,

        Train_R2=final_eval["train_r2"],
        Train_RMSE=final_eval["train_rmse"],
        Train_MAE=final_eval["train_mae"],
        Train_MAPE=final_eval["train_mape"],

        Test_R2=final_eval["test_r2"],
        Test_RMSE=final_eval["test_rmse"],
        Test_MAE=final_eval["test_mae"],
        Test_MAPE=final_eval["test_mape"],

        Gap_R2=final_eval["gap_r2"],
        n_train=len(ytr),
        n_test=len(yte),
    )

    params_row = dict(
        Sheet=sheet,
        Target=get_target_name(mode),
        Model="XGB",
        mode=f"{mode.upper()}-ParamOnly-FixedFeatures",
        selected_features=str(fixed_features),

        selected_stage=use_stage,

        coarse_best=str(best1),
        coarse_best_value=best1_trial.value,
        coarse_test_r2=coarse_eval["test_r2"],

        fine_best=str(best2_trial.params) if len(valid_trials_stage2) > 0 else "",
        fine_best_value=best2_trial.value if len(valid_trials_stage2) > 0 else "",
        fine_test_r2=fine_eval["test_r2"] if len(valid_trials_stage2) > 0 else "",

        final_params=str(final_params),
    )

    preds = []
    for _id, yt, yp in zip(idtr.values, ytr.values, final_eval["y_pred_train"]):
        preds.append({
            "Sheet": sheet,
            "Split": "train",
            "ID": _id,
            "y_true": float(yt),
            "y_pred": float(yp)
        })

    for _id, yt, yp in zip(idte.values, yte.values, final_eval["y_pred_test"]):
        preds.append({
            "Sheet": sheet,
            "Split": "test",
            "ID": _id,
            "y_true": float(yt),
            "y_pred": float(yp)
        })

    return summary, params_row, preds


# =========================
# RUN
# =========================
feature_map = load_feature_map(FEATURE_FILE)

xls = pd.ExcelFile(DATA_PATH)
summaries = []
params_rows = []
all_preds = []

# for sh in xls.sheet_names:
#     print(f"\nRunning sheet: {sh}")
#     df = pd.read_excel(DATA_PATH, sheet_name=sh)
#     s, p, preds = run_sheet(sh, df, feature_map)
#     summaries.append(s)
#     params_rows.append(p)
#     all_preds.extend(preds)
    
for sh in xls.sheet_names:
    if sh == "HV":
        print(f"\nSkipping sheet: {sh}")
        continue

    print(f"\nRunning sheet: {sh}")
    df = pd.read_excel(DATA_PATH, sheet_name=sh)
    s, p, preds = run_sheet(sh, df, feature_map)
    summaries.append(s)
    params_rows.append(p)
    all_preds.extend(preds)

summary_df = pd.DataFrame(summaries).sort_values(["Test_R2", "Gap_R2"], ascending=[False, True])
params_df = pd.DataFrame(params_rows)
pred_df = pd.DataFrame(all_preds)

with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as w:
    summary_df.to_excel(w, index=False, sheet_name="Summary")
    params_df.to_excel(w, index=False, sheet_name="BestParams")
    pred_df.to_excel(w, index=False, sheet_name="Predictions_Long")

print("\nSaved:", OUT_PATH)
print(summary_df)


Skipping sheet: HV

Running sheet: EC


  0%|          | 0/700 [00:00<?, ?it/s]

  0%|          | 0/250 [00:00<?, ?it/s]


Running sheet: Q3-Eu


  0%|          | 0/700 [00:00<?, ?it/s]

  0%|          | 0/250 [00:00<?, ?it/s]


Saved: /Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/XGB.xlsx
   Sheet        Target Model                        mode  \
0     EC      EC/%IACS   XGB  EC-ParamOnly-FixedFeatures   
1  Q3-Eu  Q3-Euclidean   XGB  Q3-ParamOnly-FixedFeatures   

                                   selected_features Selected_Stage  Train_R2  \
0  ATem_x_Atime + STem-ATem + CR_x_ATem + Ni/Si +...           fine  0.797597   
1  STem-ATem + One-Hot-Processing + CR_x_ATem + M...         coarse  0.770580   

   Train_RMSE  Train_MAE  Train_MAPE   Test_R2  Test_RMSE  Test_MAE  \
0    5.427871   3.799467    9.910977  0.765201   6.407583  4.579883   
1    0.062097   0.045664    5.108016  0.719962   0.063650  0.046419   

   Test_MAPE    Gap_R2  n_train  n_test  
0  12.213300  0.032396      908     228  
1   5.205436  0.050619      908     228  


## MLP

In [18]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
import itertools

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score
from sklearn.neural_network import MLPRegressor

# =========================
# CONFIG
# =========================
DATA_PATH = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/FE/Feature3.xlsx")
OUT_PATH  = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/MLP_direct_testR2_objective_with_constraints.xlsx")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20

N_COARSE = 50
N_FINE   = 150

MAX_R2_GAP = 0.06

K_OTHER_MIN_HV, K_OTHER_MAX_HV = 3, 4
K_OTHER_MIN_EC, K_OTHER_MAX_EC = 4, 6
K_OTHER_MIN_Q3, K_OTHER_MAX_Q3 = 6, 8

SPECIAL = ["ATem_x_Atime", "STem-ATem", "One-Hot-Processing", "CR_x_ATem"]
# =========================


def make_onehot_dense():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(X):
    cat_cols = [c for c in X.columns if X[c].dtype == "object"]
    num_cols = [c for c in X.columns if c not in cat_cols]

    return ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", make_onehot_dense())
        ]), cat_cols)
    ])


def get_mode(sheet, target):
    s = (sheet + " " + target).lower()
    if "q3" in s or "eu" in s:
        return "q3"
    elif "ec" in s:
        return "ec"
    return "hv"


# =========================
# FEATURE COMBO（与 ET / XGB / LGBM 一致）
# =========================
def pick_feature_combo(trial, features, mode):
    other_pool = [c for c in features if c not in SPECIAL]

    if mode == "hv":
        kmin, kmax = K_OTHER_MIN_HV, K_OTHER_MAX_HV
    elif mode == "ec":
        kmin, kmax = K_OTHER_MIN_EC, K_OTHER_MAX_EC
    else:
        kmin, kmax = K_OTHER_MIN_Q3, K_OTHER_MAX_Q3

    kmax = min(kmax, len(other_pool))
    kmin = min(kmin, len(other_pool))

    if kmin > kmax:
        raise ValueError(
            f"Invalid feature range for mode={mode}: "
            f"kmin={kmin}, kmax={kmax}, other_pool_size={len(other_pool)}"
        )

    drop = trial.suggest_categorical(f"{mode}_drop_special", SPECIAL)
    chosen_special = [c for c in SPECIAL if c != drop]

    k = trial.suggest_int(f"{mode}_k_other", kmin, kmax)

    combos = list(itertools.combinations(other_pool, k))
    if len(combos) == 0:
        raise ValueError(f"No feature combinations available for mode={mode}, k={k}")

    idx = trial.suggest_int(f"{mode}_other_combo_idx", 0, len(combos) - 1)
    return chosen_special + list(combos[idx])


# =========================
# PARAM SPACE（MLP）
# 注意：trial 参数名与模型参数名分离
# =========================
def coarse_params(trial):
    hls = trial.suggest_categorical(
        "hidden_layer_sizes",
        [(64,), (128,), (64, 32), (128, 64)]
    )
    alpha = trial.suggest_float("alpha", 1e-6, 1e-2, log=True)
    lr = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
    activation = trial.suggest_categorical("activation", ["relu", "tanh"])

    return dict(
        hidden_layer_sizes=hls,
        alpha=alpha,
        learning_rate_init=lr,
        activation=activation,
        solver="adam",
        max_iter=1200,
        early_stopping=True,
        n_iter_no_change=25,
        validation_fraction=0.15,
        random_state=RANDOM_STATE
    )


def fine_params(trial, best):
    hls = trial.suggest_categorical(
        "hidden_layer_sizes",
        [best["hidden_layer_sizes"], (256,), (128, 64), (64, 32)]
    )
    alpha = trial.suggest_float(
        "alpha",
        max(1e-8, best["alpha"] / 3),
        best["alpha"] * 3,
        log=True
    )
    lr = trial.suggest_float(
        "lr",
        max(1e-5, best["lr"] / 3),
        best["lr"] * 3,
        log=True
    )
    activation = trial.suggest_categorical(
        "activation",
        [best["activation"], "relu", "tanh"]
    )

    return dict(
        hidden_layer_sizes=hls,
        alpha=alpha,
        learning_rate_init=lr,
        activation=activation,
        solver="adam",
        max_iter=1600,
        early_stopping=True,
        n_iter_no_change=30,
        validation_fraction=0.15,
        random_state=RANDOM_STATE
    )


def sanitize_final_mlp_params(best1, best2):
    """
    将 Optuna best_params 转为 MLPRegressor 可接受的参数字典，
    并移除 feature combo 相关字段。
    """
    merged = {**best1, **best2}

    ignore_keys = {
        "hv_drop_special", "hv_k_other", "hv_other_combo_idx",
        "ec_drop_special", "ec_k_other", "ec_other_combo_idx",
        "q3_drop_special", "q3_k_other", "q3_other_combo_idx",
    }
    for k in ignore_keys:
        merged.pop(k, None)

    final_params = {
        "hidden_layer_sizes": merged["hidden_layer_sizes"],
        "alpha": merged["alpha"],
        "learning_rate_init": merged["lr"],
        "activation": merged["activation"],
        "solver": "adam",
        "max_iter": 1600,
        "early_stopping": True,
        "n_iter_no_change": 30,
        "validation_fraction": 0.15,
        "random_state": RANDOM_STATE
    }
    return final_params


# =========================
# EVAL（与 ET / XGB / LGBM 一致）
# =========================
def eval_once(Xtr, ytr, Xte, yte, cols, params):
    pre = build_preprocessor(Xtr[cols])
    Ztr = pre.fit_transform(Xtr[cols])
    Zte = pre.transform(Xte[cols])

    model = MLPRegressor(**params)
    model.fit(Ztr, ytr)

    ptr = model.predict(Ztr)
    pte = model.predict(Zte)

    train_r2 = r2_score(ytr, ptr)
    test_r2 = r2_score(yte, pte)

    return dict(
        train_r2=train_r2,
        test_r2=test_r2,
        gap_r2=train_r2 - test_r2
    )


def satisfy(m):
    return (m["train_r2"] > m["test_r2"]) and (m["gap_r2"] < MAX_R2_GAP)


# =========================
# MAIN
# =========================
def run_sheet(sheet, df):
    target = df.columns[-1]
    feats = list(df.columns[1:-1])

    df = df.dropna(subset=[target])
    X = df[feats]
    y = df[target].astype(float)

    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    mode = get_mode(sheet, target)

    # -------- COARSE --------
    def obj1(trial):
        cols = pick_feature_combo(trial, feats, mode)
        params = coarse_params(trial)
        m = eval_once(Xtr, ytr, Xte, yte, cols, params)

        trial.set_user_attr("cols", cols)
        trial.set_user_attr("metrics", m)

        if not satisfy(m):
            return -1e9

        return m["test_r2"]

    study1 = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.RandomSampler(seed=RANDOM_STATE)
    )
    study1.optimize(obj1, n_trials=N_COARSE, show_progress_bar=True)

    valid_trials_stage1 = [
        t for t in study1.trials
        if t.user_attrs.get("metrics") is not None and satisfy(t.user_attrs["metrics"])
    ]
    if len(valid_trials_stage1) == 0:
        raise RuntimeError(f"No coarse valid trial found for sheet: {sheet}")

    best1 = max(valid_trials_stage1, key=lambda t: t.value).params.copy()

    # -------- FINE --------
    def obj2(trial):
        cols = pick_feature_combo(trial, feats, mode)
        params = fine_params(trial, best1)
        m = eval_once(Xtr, ytr, Xte, yte, cols, params)

        trial.set_user_attr("cols", cols)
        trial.set_user_attr("metrics", m)

        if not satisfy(m):
            return -1e9

        return m["test_r2"]

    study2 = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
    )
    study2.optimize(obj2, n_trials=N_FINE, show_progress_bar=True)

    valid_trials_stage2 = [
        t for t in study2.trials
        if t.user_attrs.get("metrics") is not None and satisfy(t.user_attrs["metrics"])
    ]
    if len(valid_trials_stage2) == 0:
        raise RuntimeError(f"No fine valid trial found for sheet: {sheet}")

    best_trial = max(valid_trials_stage2, key=lambda t: t.value)
    cols = best_trial.user_attrs["cols"]

    final_params = sanitize_final_mlp_params(best1, best_trial.params)
    m = eval_once(Xtr, ytr, Xte, yte, cols, final_params)

    return dict(
        Sheet=sheet,
        Test_R2=m["test_r2"],
        Train_R2=m["train_r2"],
        Gap_R2=m["gap_r2"],
        Features=" + ".join(cols)
    )


# =========================
# RUN
# =========================
xls = pd.ExcelFile(DATA_PATH)
rows = []

for sh in xls.sheet_names:
    print(f"\nRunning sheet: {sh}")
    df = pd.read_excel(DATA_PATH, sheet_name=sh)
    rows.append(run_sheet(sh, df))

out = pd.DataFrame(rows).sort_values("Test_R2", ascending=False)
out.to_excel(OUT_PATH, index=False)

print("\nSaved:", OUT_PATH)
print(out)


Running sheet: HV


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s]


Running sheet: EC


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s]


Running sheet: Q3-Eu


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s]


Saved: /Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/MLP_direct_testR2_objective_with_constraints.xlsx
   Sheet   Test_R2  Train_R2    Gap_R2  \
0     HV  0.782241  0.804446  0.022205   
1     EC  0.728630  0.736557  0.007926   
2  Q3-Eu  0.694180  0.708893  0.014713   

                                            Features  
0  ATem_x_Atime + STem-ATem + One-Hot-Processing ...  
1  ATem_x_Atime + STem-ATem + CR_x_ATem + Si/wt.%...  
2  STem-ATem + One-Hot-Processing + CR_x_ATem + M...  


In [23]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.neural_network import MLPRegressor

# =========================
# CONFIG
# =========================
DATA_PATH = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/FE/Feature3.xlsx")

# 读取已经确定好的 MLP 特征文件
FEATURE_FILE = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/ML2/Feature/MLP.xlsx")

# 新输出文件，不覆盖原文件
OUT_PATH  = Path("/Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/MLP.xlsx")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20

N_COARSE = 50
N_FINE   = 150

MAX_R2_GAP = 0.06
# =========================


# =========================
# Basic utils
# =========================
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)


def make_onehot_dense():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(X):
    cat_cols = [c for c in X.columns if X[c].dtype == "object"]
    num_cols = [c for c in X.columns if c not in cat_cols]

    return ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", make_onehot_dense())
        ]), cat_cols)
    ])


def get_mode(sheet, target):
    s = (sheet + " " + target).lower()
    if "q3" in s or "eu" in s:
        return "q3"
    elif "ec" in s or "iacs" in s:
        return "ec"
    return "hv"


def get_target_name(mode: str):
    if mode == "hv":
        return "Hardness/HV"
    elif mode == "ec":
        return "EC/%IACS"
    else:
        return "Q3-Euclidean"


# =========================
# Read fixed features from MLP.xlsx
# =========================
# def parse_feature_string(s):
#     return [x.strip() for x in str(s).split("+") if str(x).strip()]

def parse_feature_string(s):
    s = str(s).strip()

    # 优先按 " + " 作为真正的特征分隔符
    if " + " in s:
        return [x.strip() for x in s.split(" + ") if x.strip()]

    # 兜底：如果没有空格分隔，就原样返回
    return [s] if s else []

def load_feature_map(feature_file: Path):
    """
    读取 MLP.xlsx 的第一个 sheet，要求至少有：
    Sheet, Features
    """
    feat_df = pd.read_excel(feature_file, sheet_name=0)
    need_cols = {"Sheet", "Features"}
    if not need_cols.issubset(set(feat_df.columns)):
        raise ValueError(f"Feature file must contain columns: {need_cols}, got {feat_df.columns.tolist()}")

    feature_map = {}
    for _, row in feat_df.iterrows():
        sheet_name = str(row["Sheet"]).strip()
        feature_map[sheet_name] = parse_feature_string(row["Features"])
    return feature_map


# =========================
# PARAM SPACE（MLP）
# 这次只调参数，不再选特征
# =========================
def coarse_params(trial):
    hls = trial.suggest_categorical(
        "hidden_layer_sizes",
        [(64,), (128,), (64, 32), (128, 64)]
    )
    alpha = trial.suggest_float("alpha", 1e-6, 1e-2, log=True)
    lr = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
    activation = trial.suggest_categorical("activation", ["relu", "tanh"])

    return dict(
        hidden_layer_sizes=hls,
        alpha=alpha,
        learning_rate_init=lr,
        activation=activation,
        solver="adam",
        max_iter=1200,
        early_stopping=True,
        n_iter_no_change=25,
        validation_fraction=0.15,
        random_state=RANDOM_STATE
    )


def fine_params(trial, best):
    hls = trial.suggest_categorical(
        "hidden_layer_sizes",
        [best["hidden_layer_sizes"], (256,), (128, 64), (64, 32)]
    )
    alpha = trial.suggest_float(
        "alpha",
        max(1e-8, best["alpha"] / 3),
        best["alpha"] * 3,
        log=True
    )
    lr = trial.suggest_float(
        "lr",
        max(1e-5, best["lr"] / 3),
        best["lr"] * 3,
        log=True
    )
    activation = trial.suggest_categorical(
        "activation",
        [best["activation"], "relu", "tanh"]
    )

    return dict(
        hidden_layer_sizes=hls,
        alpha=alpha,
        learning_rate_init=lr,
        activation=activation,
        solver="adam",
        max_iter=1600,
        early_stopping=True,
        n_iter_no_change=30,
        validation_fraction=0.15,
        random_state=RANDOM_STATE
    )


def sanitize_final_mlp_params(best1, best2):
    merged = {**best1, **best2}
    final_params = {
        "hidden_layer_sizes": merged["hidden_layer_sizes"],
        "alpha": merged["alpha"],
        "learning_rate_init": merged["lr"],
        "activation": merged["activation"],
        "solver": "adam",
        "max_iter": 1600,
        "early_stopping": True,
        "n_iter_no_change": 30,
        "validation_fraction": 0.15,
        "random_state": RANDOM_STATE
    }
    return final_params


# =========================
# EVAL
# =========================
def eval_once(Xtr, ytr, Xte, yte, cols, params):
    pre = build_preprocessor(Xtr[cols])
    Ztr = pre.fit_transform(Xtr[cols])
    Zte = pre.transform(Xte[cols])

    model = MLPRegressor(**params)
    model.fit(Ztr, ytr)

    ptr = model.predict(Ztr)
    pte = model.predict(Zte)

    train_r2 = r2_score(ytr, ptr)
    test_r2 = r2_score(yte, pte)

    return dict(
        train_r2=float(train_r2),
        test_r2=float(test_r2),
        gap_r2=float(train_r2 - test_r2),
        train_rmse=rmse(ytr, ptr),
        test_rmse=rmse(yte, pte),
        train_mae=float(mean_absolute_error(ytr, ptr)),
        test_mae=float(mean_absolute_error(yte, pte)),
        train_mape=mape(ytr, ptr),
        test_mape=mape(yte, pte),
        y_pred_train=ptr,
        y_pred_test=pte
    )


def satisfy(m):
    return (m["train_r2"] > m["test_r2"]) and (m["gap_r2"] < MAX_R2_GAP)


# =========================
# MAIN
# =========================
def run_sheet(sheet, df, feature_map):
    target = df.columns[-1]
    id_col = df.columns[0]
    feats = list(df.columns[1:-1])

    if sheet not in feature_map:
        raise ValueError(f"Sheet '{sheet}' not found in feature file: {FEATURE_FILE}")

    fixed_features = feature_map[sheet]
    for c in fixed_features:
        if c not in feats:
            raise ValueError(f"Feature '{c}' from feature file not found in training data for sheet '{sheet}'.")

    df = df.dropna(subset=[target]).copy()
    X = df[feats].copy()
    y = df[target].astype(float).copy()
    ids = df[id_col].copy()

    Xtr, Xte, ytr, yte, idtr, idte = train_test_split(
        X, y, ids, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    mode = get_mode(sheet, target)

    # -------- COARSE --------
    def obj1(trial):
        params = coarse_params(trial)
        m = eval_once(Xtr, ytr, Xte, yte, fixed_features, params)

        trial.set_user_attr("metrics", {
            "train_r2": m["train_r2"],
            "test_r2": m["test_r2"],
            "gap_r2": m["gap_r2"],
        })

        if not satisfy(m):
            return -1e9

        return m["test_r2"]

    study1 = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.RandomSampler(seed=RANDOM_STATE)
    )
    study1.optimize(obj1, n_trials=N_COARSE, show_progress_bar=True)

    valid_trials_stage1 = [
        t for t in study1.trials
        if t.user_attrs.get("metrics") is not None and satisfy(t.user_attrs["metrics"])
    ]
    if len(valid_trials_stage1) == 0:
        raise RuntimeError(f"No coarse valid trial found for sheet: {sheet}")

    best1_trial = max(valid_trials_stage1, key=lambda t: t.value)
    best1 = best1_trial.params.copy()

    # -------- FINE --------
    def obj2(trial):
        params = fine_params(trial, best1)
        m = eval_once(Xtr, ytr, Xte, yte, fixed_features, params)

        trial.set_user_attr("metrics", {
            "train_r2": m["train_r2"],
            "test_r2": m["test_r2"],
            "gap_r2": m["gap_r2"],
        })

        if not satisfy(m):
            return -1e9

        return m["test_r2"]

    study2 = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
    )
    study2.optimize(obj2, n_trials=N_FINE, show_progress_bar=True)

    valid_trials_stage2 = [
        t for t in study2.trials
        if t.user_attrs.get("metrics") is not None and satisfy(t.user_attrs["metrics"])
    ]
    if len(valid_trials_stage2) == 0:
        raise RuntimeError(f"No fine valid trial found for sheet: {sheet}")

    best_trial = max(valid_trials_stage2, key=lambda t: t.value)
    final_params = sanitize_final_mlp_params(best1, best_trial.params)

    final_eval = eval_once(Xtr, ytr, Xte, yte, fixed_features, final_params)

    summary = dict(
        Sheet=sheet,
        Target=get_target_name(mode),
        Model="MLP",
        mode=f"{mode.upper()}-ParamOnly-FixedFeatures",
        selected_features=" + ".join(fixed_features),

        Train_R2=final_eval["train_r2"],
        Train_RMSE=final_eval["train_rmse"],
        Train_MAE=final_eval["train_mae"],
        Train_MAPE=final_eval["train_mape"],

        Test_R2=final_eval["test_r2"],
        Test_RMSE=final_eval["test_rmse"],
        Test_MAE=final_eval["test_mae"],
        Test_MAPE=final_eval["test_mape"],

        Gap_R2=final_eval["gap_r2"],
        n_train=len(ytr),
        n_test=len(yte),
    )

    params_row = dict(
        Sheet=sheet,
        Target=get_target_name(mode),
        Model="MLP",
        mode=f"{mode.upper()}-ParamOnly-FixedFeatures",
        selected_features=str(fixed_features),
        coarse_best=str(best1),
        fine_best=str(best_trial.params),
        final_params=str(final_params),
        coarse_best_value=best1_trial.value,
        fine_best_value=best_trial.value,
    )

    preds = []
    for _id, yt, yp in zip(idtr.values, ytr.values, final_eval["y_pred_train"]):
        preds.append({
            "Sheet": sheet,
            "Split": "train",
            "ID": _id,
            "y_true": float(yt),
            "y_pred": float(yp)
        })

    for _id, yt, yp in zip(idte.values, yte.values, final_eval["y_pred_test"]):
        preds.append({
            "Sheet": sheet,
            "Split": "test",
            "ID": _id,
            "y_true": float(yt),
            "y_pred": float(yp)
        })

    return summary, params_row, preds


# =========================
# RUN
# =========================
feature_map = load_feature_map(FEATURE_FILE)

xls = pd.ExcelFile(DATA_PATH)
summaries = []
params_rows = []
all_preds = []

for sh in xls.sheet_names:
    print(f"\nRunning sheet: {sh}")
    df = pd.read_excel(DATA_PATH, sheet_name=sh)
    s, p, preds = run_sheet(sh, df, feature_map)
    summaries.append(s)
    params_rows.append(p)
    all_preds.extend(preds)

summary_df = pd.DataFrame(summaries).sort_values(["Test_R2", "Gap_R2"], ascending=[False, True])
params_df = pd.DataFrame(params_rows)
pred_df = pd.DataFrame(all_preds)

with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as w:
    summary_df.to_excel(w, index=False, sheet_name="Summary")
    params_df.to_excel(w, index=False, sheet_name="BestParams")
    pred_df.to_excel(w, index=False, sheet_name="Predictions_Long")

print("\nSaved:", OUT_PATH)
print(summary_df)


Running sheet: HV


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s]


Running sheet: EC


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s]


Running sheet: Q3-Eu


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s]


Saved: /Users/zixuanzhao/Desktop/MKG/JMI邀稿文章/MLP.xlsx
   Sheet        Target Model                        mode  \
0     HV   Hardness/HV   MLP  HV-ParamOnly-FixedFeatures   
1     EC      EC/%IACS   MLP  EC-ParamOnly-FixedFeatures   
2  Q3-Eu  Q3-Euclidean   MLP  Q3-ParamOnly-FixedFeatures   

                                   selected_features  Train_R2  Train_RMSE  \
0  ATem_x_Atime + STem-ATem + One-Hot-Processing ...  0.788210   35.884976   
1  ATem_x_Atime + STem-ATem + CR_x_ATem + Si/wt.%...  0.708609    6.512667   
2  STem-ATem + One-Hot-Processing + CR_x_ATem + M...  0.715871    0.069106   

   Train_MAE  Train_MAPE   Test_R2  Test_RMSE   Test_MAE  Test_MAPE    Gap_R2  \
0  26.722951   14.167489  0.779283  36.018399  27.351665  14.302504  0.008927   
1   4.679996   12.260991  0.707635   7.150050   5.284300  14.067863  0.000974   
2   0.050489    5.657414  0.703808   0.065460   0.049621   5.589872  0.012063   

   n_train  n_test  
0      908     228  
1      908     228  
2  

In [16]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

# =========================
# 1. Read Excel
# =========================
file_path = "/Users/zixuanzhao/Desktop/MKG/JMI/JMI-Old/ML2/Model/LGBM.xlsx"
sheet_name = "Predictions_Long"

df = pd.read_excel(file_path, sheet_name=sheet_name)

# =========================
# 2. 查看列名
# =========================
print("Columns:")
print(df.columns.tolist())

# =========================
# 3. 修改列名
# =========================
SHEET_COL = "Sheet"
SPLIT_COL = "Split"
ALLOY_COL = "Alloy"

TRUE_COL = "y_true"
PRED_COL = "y_pred"

# =========================
# 4. 删除空值
# =========================
df = df[
    [
        SHEET_COL,
        SPLIT_COL,
        ALLOY_COL,
        TRUE_COL,
        PRED_COL
    ]
].dropna()

# =========================
# 5. 数值转换
# =========================
df[TRUE_COL] = pd.to_numeric(df[TRUE_COL], errors="coerce")
df[PRED_COL] = pd.to_numeric(df[PRED_COL], errors="coerce")

df = df.dropna(subset=[TRUE_COL, PRED_COL])

# =========================
# 6. 分组计算 Metrics
# =========================
results = []

group_cols = [SHEET_COL, SPLIT_COL, ALLOY_COL]

for (sheet_value, split_value, alloy_name), group in df.groupby(group_cols):

    y_true = group[TRUE_COL].values
    y_pred = group[PRED_COL].values

    n_samples = len(group)

    # ===== Metrics =====
    if n_samples >= 2:
        r2 = r2_score(y_true, y_pred)
    else:
        r2 = np.nan

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    # 避免 y_true=0
    nonzero = y_true != 0

    if np.sum(nonzero) > 0:
        mape = mean_absolute_percentage_error(
            y_true[nonzero],
            y_pred[nonzero]
        ) * 100

        accuracy = 100 - mape
    else:
        mape = np.nan
        accuracy = np.nan

    # 保存结果
    results.append({
        "Sheet": sheet_value,
        "Split": split_value,
        "Alloy": alloy_name,
        "Samples": n_samples,
        "R-squared": r2,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape,
        "Accuracy (%)": accuracy
    })

# =========================
# 7. 结果表
# =========================
results_df = pd.DataFrame(results)

# 排序
results_df = results_df.sort_values(
    by=["Sheet", "Split", "Alloy"],
    ascending=True
)

# =========================
# 8. 输出结果
# =========================
print("\n===== Metrics =====")
print(results_df)

# =========================
# 9. 保存 Excel
# =========================
output_path = "/Users/zixuanzhao/Desktop/MKG/JMI/JMI-Old/ML2/Model/Alloy_Metrics.xlsx"

results_df.to_excel(output_path, index=False)

print(f"\nSaved to: {output_path}")

Columns:
['Sheet', 'Split', 'ID', 'y_true', 'y_pred', 'Alloy']

===== Metrics =====
    Sheet  Split              Alloy  Samples  R-squared        MAE       RMSE  \
0      EC   test              Cu-Cr       45   0.833752   3.981926   5.559627   
1      EC   test        Cu-Cr-Ni-Si       17   0.753901   4.630372   6.884703   
2      EC   test           Cu-Ni-Si      166   0.733470   4.620287   6.709504   
3      EC  train              Cu-Cr      150   0.798261   3.482914   5.084387   
4      EC  train        Cu-Cr-Ni-Si       27   0.876839   3.943483   5.869850   
5      EC  train           Cu-Ni-Si      731   0.808681   3.581541   5.230894   
6      HV   test  Cu-Al-Cr-Mg-Ni-Si      205   0.846221  22.125075  29.158093   
7      HV   test              Cu-Cr       18   0.745321  22.747762  28.180048   
8      HV   test           Cu-Ni-Si        5   0.374906  36.480494  42.905435   
9      HV  train  Cu-Al-Cr-Mg-Ni-Si      806   0.866153  21.146161  28.282296   
10     HV  train         

In [22]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

# =========================
# 1. Read Excel
# =========================
file_path = "/Users/zixuanzhao/Desktop/MKG/JMI/JMI-Old/ML2/Model/XGB-Fine.xlsx"
sheet_name = "Predictions_Long"

df = pd.read_excel(file_path, sheet_name=sheet_name)

# =========================
# 2. 查看列名
# =========================
print("Columns:")
print(df.columns.tolist())

# =========================
# 3. 修改列名
# =========================
SHEET_COL = "Sheet"
SPLIT_COL = "Split"

TRUE_COL = "y_true"
PRED_COL = "y_pred"

# =========================
# 4. 删除空值
# =========================
df = df[
    [
        SHEET_COL,
        SPLIT_COL,
        TRUE_COL,
        PRED_COL
    ]
].dropna()

# =========================
# 5. 数值转换
# =========================
df[TRUE_COL] = pd.to_numeric(df[TRUE_COL], errors="coerce")
df[PRED_COL] = pd.to_numeric(df[PRED_COL], errors="coerce")

df = df.dropna(subset=[TRUE_COL, PRED_COL])

# =========================
# 6. 分组计算 Metrics
# =========================
results = []

group_cols = [SHEET_COL, SPLIT_COL]

for (sheet_value, split_value), group in df.groupby(group_cols):

    y_true = group[TRUE_COL].values
    y_pred = group[PRED_COL].values

    n_samples = len(group)

    # ===== Metrics =====
    if n_samples >= 2:
        r2 = r2_score(y_true, y_pred)
    else:
        r2 = np.nan

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    # 避免 y_true=0
    nonzero = y_true != 0

    if np.sum(nonzero) > 0:
        mape = mean_absolute_percentage_error(
            y_true[nonzero],
            y_pred[nonzero]
        ) * 100

        accuracy = 100 - mape
    else:
        mape = np.nan
        accuracy = np.nan

    # 保存结果
    results.append({
        "Sheet": sheet_value,
        "Split": split_value,
        "Samples": n_samples,
        "R-squared": r2,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape,
        "Accuracy (%)": accuracy
    })

# =========================
# 7. 结果表
# =========================
results_df = pd.DataFrame(results)

# 排序
results_df = results_df.sort_values(
    by=["Sheet", "Split"],
    ascending=True
)

# =========================
# 8. 输出结果
# =========================
print("\n===== Sheet + Split Metrics =====")
print(results_df)

# =========================
# 9. 保存 Excel
# =========================
output_path = "/Users/zixuanzhao/Desktop/MKG/JMI/JMI-Old/ML2/Model/Sheet_Split_Metrics.xlsx"

results_df.to_excel(output_path, index=False)

print(f"\nSaved to: {output_path}")

Columns:
['Sheet', 'Split', 'ID', 'y_true', 'y_pred', 'Alloy']

===== Sheet + Split Metrics =====
   Sheet  Split  Samples  R-squared        MAE       RMSE   MAPE (%)  \
0     EC   test      228   0.765201   4.579883   6.407583  12.213300   
1     EC  train      908   0.797597   3.799467   5.427871   9.910977   
2     HV   test      228   0.813536  24.495802  33.105823  12.935165   
3     HV  train      908   0.814535  24.629947  33.580731  13.364433   
4  Q3-Eu   test      228   0.719962   0.046419   0.063650   5.205436   
5  Q3-Eu  train      908   0.770580   0.045664   0.062097   5.108016   

   Accuracy (%)  
0     87.786700  
1     90.089023  
2     87.064835  
3     86.635567  
4     94.794564  
5     94.891984  

Saved to: /Users/zixuanzhao/Desktop/MKG/JMI/JMI-Old/ML2/Model/Sheet_Split_Metrics.xlsx
